# Exploratory analysis notebook (original)

The statistics reported in the paper are produced by the R scripts in `analysis/`. This notebook is the original exploratory Python analysis, kept for transparency. Outputs were removed because they contained participant-level results, and data paths point to the synthetic dataset. The second cell is an experimental LPFG variant (Gemini 2.0 Flash, run in Colab, with the game source passed as text); it is not the deployed system, which is `lpfg/adviser.py`.


In [ ]:
!pip install websockets

In [ ]:
import asyncio
import websockets
import json
import threading
import socket
from multiprocessing import Process, Queue
import cv2
import numpy as np
from cv2 import COLOR_BGRA2BGR, cvtColor
from numpy import asarray
import csv
import time
from datetime import datetime, timedelta
import pandas as pd
import os
import time
# ===== Google Generative AI Integration =====
import google.generativeai as genai
import typing_extensions as typing

# ===== Flask for Serving Advice to JS =====

# Global variable to store the latest advice
advice_data = {
    "advices": []
}
genai.configure(
    api_key=os.environ["GOOGLE_API_KEY"])
model_name = "models/gemini-2.0-flash-exp"
# cache = genai.caching.CachedContent.create(
#     model=model_name,
#     system_instruction=(
# ,
#     contents=[document_expert]
# )
CSV_PATH = r"/content/gaze_data/gaze_data_4.csv"
def generate_advice():
    global advice_data  # Declare that we are modifying the global advice_data
    global CSV_PATH
    current_time = time.time()
    genai.configure(
        api_key=os.environ["GOOGLE_API_KEY"])
    document_expert = genai.upload_file(path=r"gaze_data/gaze_sequences_expert.csv",mime_type =  'text/csv')

    # Process the CSV data and call the Generative AI model

    sequences_csv_path = CSV_PATH.replace("gaze_data_", "gaze_sequences_")
    process_gaze_data(CSV_PATH, sequences_csv_path)
    model = genai.GenerativeModel(model_name)

    document_novice = genai.upload_file(path=sequences_csv_path,mime_type =  'text/csv')
    response = model.generate_content(
        contents=[        "You are an expert analyzing gaze data. "
        "I will provide you with the expert gaze data. Ignore the first and second column. "
        "The third column includes the name of the information box and the fourth column is the duration of the gaze. "
        "It is important to note that the data is ordered chronologically. ",document_expert,
        "Now based on this data, I will provide you with novice data and ask you to analyze it "
        "and provide one or two advices to the novice user. Please be concise and provide advice directly to the novice user. ",
        "Please use the game code to give more practical advice",
        """
// RestaurantSimulation.js
import React, { useState, useEffect, useRef } from 'react';
import {
  Card,
  CardContent,
  CardHeader,
  Button,
  Select,
  SelectContent,
  SelectItem,
  SelectTrigger,
  SelectValue,
  Checkbox,
  Input,
} from './components';
import { Rnd } from 'react-rnd';
import KitchenGUI from './KitchenGUI';
import { Bell, Play, Trash2, Pause, Save, Download, Settings } from 'lucide-react';
import StatsTracker from './StatsTracker';

const stats = new StatsTracker();

const INGREDIENTS = [
  { name: 'BEEF', deliveryTime: 10, cost: 5 },
  { name: 'CHICKEN', deliveryTime: 8, cost: 4 },
  { name: 'PASTA', deliveryTime: 5, cost: 2 },
  { name: 'TOMATO', deliveryTime: 3, cost: 1 },
  { name: 'CHEESE', deliveryTime: 4, cost: 3 },
  { name: 'LETTUCE', deliveryTime: 2, cost: 1 },
];

const MENU_ITEMS = [
  { name: 'Burger', ingredients: ['BEEF', 'LETTUCE', 'TOMATO'], cookingTime: 10, price: 10 },
  { name: 'Pasta', ingredients: ['PASTA', 'TOMATO', 'CHEESE'], cookingTime: 15, price: 12 },
  { name: 'Salad', ingredients: ['LETTUCE', 'TOMATO', 'CHEESE'], cookingTime: 5, price: 8 },
  { name: 'Chicken Sandwich', ingredients: ['CHICKEN', 'LETTUCE', 'TOMATO'], cookingTime: 12, price: 11 },
  { name: 'Pizza', ingredients: ['TOMATO', 'CHEESE'], cookingTime: 20, price: 15 },
];

class Random {
  constructor(seed = 12345) {
    this.seed = seed;
  }
  random() {
    const x = Math.sin(this.seed++) * 10000;
    return x - Math.floor(x);
  }
  exponential(lambda) {
    return -Math.log(1 - this.random()) / lambda;
  }
}

const MAX_WAIT_TIME = 90;
const PENALTY = 20;

const RestaurantSimulation = () => {
  const [inventory, setInventory] = useState(() =>
    Object.fromEntries(INGREDIENTS.map((ing) => [ing.name, 50]))
  );
  const [cooks, setCooks] = useState([
    { name: 'Cook 1', busy: false, currentOrder: null },
    { name: 'Cook 2', busy: false, currentOrder: null },
    { name: 'Cook 3', busy: false, currentOrder: null },
    { name: 'Cook 4', busy: false, currentOrder: null },
  ]);
  const [groups, setGroups] = useState([]);
  const [orders, setOrders] = useState([]);
  const [time, setTime] = useState(0);
  const [logs, setLogs] = useState([]);
  const [isRunning, setIsRunning] = useState(false);
  const [orderIngredient, setOrderIngredient] = useState('');
  const [orderAmount, setOrderAmount] = useState(0);
  const [incomingDeliveries, setIncomingDeliveries] = useState([]);
  const [money, setMoney] = useState(0);
  const [groupCounter, setGroupCounter] = useState(1);
  const [alerts, setAlerts] = useState([]);
  const [busyButtonStates, setBusyButtonStates] = useState({});
  const [useFixedLayout, setUseFixedLayout] = useState(false);
  const [showConfig, setShowConfig] = useState(false);
  const [randomSeed, setRandomSeed] = useState(12345);
  const [arrivalRate, setArrivalRate] = useState(0.15);
  const [rng] = useState(() => new Random(randomSeed));
  const [savedLayouts, setSavedLayouts] = useState([]);
  const [missedGroups, setMissedGroups] = useState(0);
  const [moneySpent, setMoneySpent] = useState(0);
  const [userId, setUserId] = useState('');
  const [selectedLayout, setSelectedLayout] = useState('');
  const [cookCost, setCookCost] = useState(100);

  useEffect(() => {
    if (!isRunning) {
      stats.saveStats();
    }
  }, [isRunning]);

  useEffect(() => {
    return () => {
      stats.saveStats();
    };
  }, []);

  useEffect(() => {
    const layouts = JSON.parse(localStorage.getItem('savedLayouts')) || [];
    setSavedLayouts(layouts);
  }, []);



//useEffect(() => {
//  let intervalId;
//
//  if (isRunning) {
//    // Fetch advice immediately when simulation starts
//    // Set interval to fetch advice every 2 minutes while simulation is running
//    intervalId = setInterval(fetchAdvice, 12000);
//  }
//
//  return () => {
//    if (intervalId) clearInterval(intervalId);
//  };
//}, [isRunning, time]);

  const renderSimulationConfig = () => {
    if (!showConfig) return null;
    const downloadStats = () => {
      const allStats = stats.getAllStats();
      if (allStats.length === 0) {
        alert('No stats available to download');
        return;
      }

      const csvHeader =
        'timestamp,userId,customersServed,groupsServed,groupsMissed,moneyMade,moneySpent\n';
      const csvContent = allStats
        .map(
          (stat) =>
            `${stat.timestamp},${stat.userId},${stat.customersServed},${stat.groupsServed},${stat.groupsMissed},${stat.moneyMade},${stat.moneySpent}`
        )
        .join('\n');

      const blob = new Blob([csvHeader + csvContent], { type: 'text/csv' });
      const url = URL.createObjectURL(blob);
      const a = document.createElement('a');
      a.href = url;
      a.download = 'restaurant_stats.csv';
      document.body.appendChild(a);
      a.click();
      document.body.removeChild(a);
      URL.revokeObjectURL(url);
    };

    return (
      <Card className="mb-4">
        <CardHeader>
          <div className="flex justify-between items-center">
            <h2 className="text-xl font-bold">Simulation Configuration</h2>
            <Button onClick={() => setShowConfig(false)} className="bg-gray-500 hover:bg-gray-600">
              Close
            </Button>
          </div>
        </CardHeader>
        <CardContent className="space-y-4">
          <div className="flex space-x-4">
            <div>
              <label className="block text-sm font-medium text-gray-700">User ID</label>
              <Input
                type="text"
                value={userId}
                onChange={(e) => {
                  setUserId(e.target.value);
                  stats.setUserId(e.target.value);
                }}
                placeholder="Enter experiment ID"
                className="mt-1"
              />
            </div>
            <div>
              <label className="block text-sm font-medium text-gray-700">Random Seed</label>
              <Input
                type="number"
                value={randomSeed}
                onChange={(e) => {
                  const newSeed = parseInt(e.target.value);
                  setRandomSeed(newSeed);
                  rng.seed = newSeed;
                }}
                className="mt-1"
              />
            </div>
            <div>
              <label className="block text-sm font-medium text-gray-700">
                Average Arrivals per Minute (λ)
              </label>
              <Input
                type="number"
                step="0.1"
                min="0.1"
                max="10"
                value={arrivalRate}
                onChange={(e) => setArrivalRate(parseFloat(e.target.value))}
                className="mt-1"
              />
            </div>
          </div>
          <div className="flex space-x-4">
            <Button onClick={downloadStats}>
              <Download className="mr-2 h-4 w-4" />
              Download Stats CSV
            </Button>
          </div>
        </CardContent>
      </Card>
    );
  };

  const saveCurrentLayout = () => {
    const layoutName = prompt('Enter a name for this layout:');
    if (layoutName) {
      const newLayout = { name: layoutName, positions: boxPositions };
      const updatedLayouts = [...savedLayouts, newLayout];
      setSavedLayouts(updatedLayouts);
      localStorage.setItem('savedLayouts', JSON.stringify(updatedLayouts));
      alert(`Layout "${layoutName}" saved successfully!`);
    }
  };

  const loadSavedLayout = () => {
    const layoutOptions = savedLayouts.map((layout) => layout.name);
    const selectedLayoutName = prompt(`Select a layout to load:\n${layoutOptions.join('\n')}`);

    if (selectedLayoutName) {
      const layout = savedLayouts.find((l) => l.name === selectedLayoutName);
      if (layout) {
        setBoxPositions(layout.positions);
        setUseFixedLayout(true);
        alert(`Layout "${selectedLayoutName}" loaded successfully!`);
      } else {
        alert('Layout not found. Please try again.');
      }
    } else {
      alert('No layout selected.');
    }
  };

  const initialLayout = {
    menuDetails: {
      width: 342,
      height: 222,
      x: 100,
      y: 159,
    },
    waitingGroups: {
      width: 342,
      height: 222,
      x: 103,
      y: 397,
    },
    status: {
      width: 342,
      height: 500,
      x: 705,
      y: 73,
    },
    pendingOrders: {
      width: 342,
      height: 836,
      x: 1059,
      y: 75,
    },
    hireCook: {
      width: 342,
      height: 222,
      x: 1412,
      y: 90,
    },
    orderIngredients: {
      width: 342,
      height: 237,
      x: 1414,
      y: 328,
    },
    alerts: {
      width: 342,
      height: 222,
      x: 1761,
      y: 91,
    },
    eventLog: {
      width: 342,
      height: 232,
      x: 1765,
      y: 329,
    },
    kitchenGUI: {
      width: 548,
      height: 427,
      x: 1436,
      y: 606,
    },
    moneyStatus: {
      width: 200,
      height: 100,
      x: 50,
      y: 50,
    },
    inventoryStatus: {
      width: 300,
      height: 200,
      x: 260,
      y: 50,
    },
  };
  const [boxPositions, setBoxPositions] = useState(initialLayout);

  const handleLayoutToggle = (checked) => {
    setUseFixedLayout(checked);
    if (checked) {
      setBoxPositions(initialLayout);
    }
  };

  const handleCookClick = (cookIndex, orderIndex) => {
    if (cooks[cookIndex].busy) {
      setBusyButtonStates((prev) => ({ ...prev, [cookIndex]: true }));
      setTimeout(() => {
        setBusyButtonStates((prev) => ({ ...prev, [cookIndex]: false }));
      }, 1000);
    } else {
      assignOrderToCook(cookIndex, orderIndex);
    }
  };

  const renderPendingOrders = () => {
    const groupedOrders = orders.reduce((acc, order) => {
      if (!acc[order.groupId]) {
        acc[order.groupId] = [];
      }
      acc[order.groupId].push(order);
      return acc;
    }, {});
    return (
      <Card>
        <CardHeader></CardHeader>
        <CardContent>
          {Object.entries(groupedOrders).map(([groupId, groupOrders]) => (
            <div key={groupId} className="mb-4">
              <h3 className="text-lg font-semibold border-b pb-1 mb-2">Group {groupId}</h3>
              {groupOrders.map((order, orderIndex) => (
                <div key={orderIndex} className="mb-3">
                  <p className="font-medium text-sm">
                    {order.name} (${order.price})
                  </p>
                  <div className="grid grid-cols-2 gap-2 mt-1">
                    {cooks.map((cook, cookIndex) => (
                      <Button
                        key={cookIndex}
                        onClick={() =>
                          handleCookClick(cookIndex, orders.findIndex((o) => o === order))
                        }
                        className={`text-sm py-1 ${
                          cook.busy ? 'bg-gray-400 hover:bg-gray-500' : ''
                        }`}
                      >
                        {busyButtonStates[cookIndex] ? 'BUSY' : `Assign ${cook.name}`}
                      </Button>
                    ))}
                  </div>
                </div>
              ))}
            </div>
          ))}
        </CardContent>
      </Card>
    );
  };

  const addLog = (message) => {
    setLogs((prevLogs) => [...prevLogs, `Time ${time}: ${message}`]);
  };
  const addAlert = (message) => {
    setAlerts((prevAlerts) => [...prevAlerts, { id: Date.now(), message, time }]);
  };
  const hireCook = () => {
    if (money >= cookCost) {
      const newCook = { name: `Cook ${cooks.length + 1}`, busy: false, currentOrder: null };
      setCooks((prevCooks) => [...prevCooks, newCook]);
      setMoney((prevMoney) => prevMoney - cookCost);
      addLog(`Hired a new cook for $${cookCost}`);
      setCookCost((prevCost) => Math.floor(prevCost * 1.5));
      stats.addMoneySpent(cookCost);
    } else {
      addAlert(`Not enough money to hire a new cook. Need $${cookCost}.`);
    }
  };

  const checkForAlerts = () => {
    // Check for low inventory
    Object.entries(inventory).forEach(([ing, count]) => {
      if (count <= 10) {
        const alertMessage = `Low inventory: Only ${count} ${ing} left!`;
        addAlert(alertMessage);
      }
    });


    groups.forEach((group) => {
      const waitTime = time - group.arrivalTime;
      if (waitTime >= 75) {
        addAlert(`Group ${group.id} has been waiting for ${waitTime} seconds!`);
      }
    });
  };

  const addGroup = () => {
    const groupSize = Math.floor(rng.random() * 6) + 1;
    const groupOrders = Array(groupSize)
      .fill()
      .map(() => ({
        ...MENU_ITEMS[Math.floor(rng.random() * MENU_ITEMS.length)],
        groupId: groupCounter,
      }));
    const newGroup = {
      id: groupCounter,
      size: groupSize,
      orders: groupOrders,
      arrivalTime: time,
      completedOrders: 0,
    };
    setGroups((prev) => [...prev, newGroup]);
    setOrders((prev) => [...prev, ...groupOrders]);
    addLog(
      `New group ${groupCounter} of ${groupSize} arrived and ordered ${groupOrders
        .map((o) => o.name)
        .join(', ')}`
    );
    setGroupCounter((prev) => prev + 1);
  };

  const assignOrderToCook = (cookIndex, orderIndex) => {
    const order = orders[orderIndex];
    if (order.ingredients.every((ing) => inventory[ing] > 0)) {
      setCooks((prev) =>
        prev.map((cook, i) =>
          i === cookIndex ? { ...cook, busy: true, currentOrder: { ...order, startTime: time } } : cook
        )
      );
      setOrders((prev) => prev.filter((_, i) => i !== orderIndex));
      addLog(`Assigned ${order.name} to ${cooks[cookIndex].name}`);
      stats.incrementCustomersServed();
    } else {
      addLog(`Not enough ingredients for ${order.name}`);
    }
  };

  const updateCooks = () => {
    setCooks((prev) =>
      prev.map((cook) => {
        if (cook.busy && cook.currentOrder) {
          cook.currentOrder.cookingTime--;
          if (cook.currentOrder.cookingTime <= 0) {
            setInventory((prevInventory) => {
              const newInventory = { ...prevInventory };
              cook.currentOrder.ingredients.forEach((ing) => newInventory[ing]--);
              return newInventory;
            });
            addLog(`${cook.name} finished ${cook.currentOrder.name}`);

            setGroups((prevGroups) => {
              const groupIndex = prevGroups.findIndex((g) => g.id === cook.currentOrder.groupId);
              if (groupIndex !== -1) {
                const group = prevGroups[groupIndex];
                const updatedGroup = {
                  ...group,
                  completedOrders: group.completedOrders + 1,
                  orders: group.orders.filter((o) => o.name !== cook.currentOrder.name),
                };

                if (updatedGroup.completedOrders === updatedGroup.size) {
                  const waitTime = time - group.arrivalTime;

                  // Calculate base earnings as the sum of all order prices in the group
                  const baseEarnings = group.orders.reduce((total, order) => total + order.price, 0);

                  // Optional: Adjust tip based on base earnings or keep it as is
                  const tipPercentage = Math.max(0, 0.05 - Math.floor(waitTime / 2000) * 0.01); // Example formula
                  const tip = baseEarnings * tipPercentage;

                  const penalty = waitTime > MAX_WAIT_TIME ? PENALTY : 0;
                  const totalEarned = baseEarnings + tip - penalty;

                  // Update money and stats
                  stats.addMoneyMade(totalEarned);
                  stats.incrementGroupsServed();

                  setMoney((prev) => prev + totalEarned);
                  addLog(
                    `Group ${group.id} of ${group.size} finished. Earned $${totalEarned} (${baseEarnings} + ${tip} tip - ${penalty} penalty)`
                  );

                  // Remove the group from the active groups
                  return prevGroups.filter((_, i) => i !== groupIndex);

                } else {
                  return [
                    ...prevGroups.slice(0, groupIndex),
                    updatedGroup,
                    ...prevGroups.slice(groupIndex + 1),
                  ];
                }
              }
              return prevGroups;
            });

            return { ...cook, busy: false, currentOrder: null };
          }
        }
        return cook;
      })
    );
  };

  const updateDeliveries = () => {
    setIncomingDeliveries((prev) => {
      const updatedDeliveries = prev.map((delivery) => ({
        ...delivery,
        timeLeft: delivery.timeLeft - 1,
      }));

      updatedDeliveries.forEach((delivery) => {
        if (delivery.timeLeft <= 0) {
          setInventory((prevInventory) => ({
            ...prevInventory,
            [delivery.ingredient]: prevInventory[delivery.ingredient] + delivery.amount,
          }));
          addLog(`Received delivery of ${delivery.amount} ${delivery.ingredient}`);
        }
      });

      return updatedDeliveries.filter((delivery) => delivery.timeLeft > 0);
    });
  };

  const updateGroups = () => {
    setGroups((prev) => {
      const updatedGroups = prev.map((group) => {
        const waitTime = time - group.arrivalTime;
        if (waitTime >= MAX_WAIT_TIME && group.completedOrders === 0) {
          stats.incrementGroupsMissed();
          addLog(`Group ${group.id} of ${group.size} left due to long wait time. Lost $${PENALTY}`);
          setMoney((prevMoney) => prevMoney - PENALTY);
          setOrders((prevOrders) => prevOrders.filter((order) => order.groupId !== group.id));
          return null;
        }

        return group;
      });
      return updatedGroups.filter((group) => group !== null);
    });
  };

  const runSimulation = () => {
    setTime((prev) => prev + 1);

    const timeUntilNextArrival = rng.exponential(arrivalRate);
    if (timeUntilNextArrival < 1) {
      addGroup();
    }

    updateCooks();
    updateDeliveries();
    updateGroups();
    checkForAlerts();
  };
useEffect(() => {
  if (savedLayouts.length > 0) {
    const layout2 = savedLayouts.find(layout => layout.name === "Layout2");
    if (layout2) {
      setBoxPositions(layout2.positions);
      setUseFixedLayout(true); // Optional: Prevent layout changes
      console.log('Layout 2 loaded automatically.');
    } else {
      console.warn('"Layout 2" not found in savedLayouts.');
      // Optionally, handle the absence of "Layout 2" here
    }
  }
}, [savedLayouts]);

  useEffect(() => {
    let timer;
    if (isRunning) {
      timer = setInterval(runSimulation, 1000);
    }
    return () => clearInterval(timer);
  }, [isRunning, runSimulation]);

  const handleOrderIngredient = () => {
    if (orderIngredient === '' || orderAmount === 0) {
      addLog('Please select both an ingredient and an amount');
      return;
    }

    const ingredient = INGREDIENTS.find((ing) => ing.name === orderIngredient);
    if (!ingredient) {
      addLog(`Error: Ingredient ${orderIngredient} not found`);
      return;
    }

    const cost = orderAmount * ingredient.cost;
    if (money >= cost) {
      stats.addMoneySpent(cost);

      setIncomingDeliveries((prev) => [
        ...prev,
        {
          ingredient: orderIngredient,
          amount: orderAmount,
          timeLeft: ingredient.deliveryTime,
        },
      ]);
      setMoney((prev) => prev - cost);
      addLog(
        `Ordered ${orderAmount} units of ${orderIngredient} for $${cost}. Estimated delivery time: ${ingredient.deliveryTime} time units`
      );
      setOrderIngredient('');
      setOrderAmount(0);
    } else {
      addLog(`Not enough money to order ${orderAmount} units of ${orderIngredient}`);
    }
  };

  const boxRefs = {
    status: useRef(null),
    waitingGroups: useRef(null),
    pendingOrders: useRef(null),
    orderIngredients: useRef(null),
    eventLog: useRef(null),
    alerts: useRef(null),
    hireCook: useRef(null),
    menuDetails: useRef(null),
    moneyStatus: useRef(null),
    inventoryStatus: useRef(null),
    kitchenGUI: useRef(null),
  };

  const [boxSizes, setBoxSizes] = useState({
    status: { width: 320, height: 200 },
    waitingGroups: { width: 320, height: 200 },
    pendingOrders: { width: 320, height: 200 },
    orderIngredients: { width: 320, height: 200 },
    eventLog: { width: 320, height: 200 },
    alerts: { width: 320, height: 200 },
    hireCook: { width: 320, height: 200 },
    menuDetails: { width: 320, height: 200 },
    moneyStatus: { width: 200, height: 100 },
    inventoryStatus: { width: 300, height: 200 },
  });

  const commonBoxStyle = {
    border: '1px solid #000',
    padding: '10px',
    backgroundColor: '#fff',
    overflow: 'auto',
    width: '100%',
    height: '100%',
  };

  const cardStyle = {
    border: '1px solid #000',
    padding: '10px',
    backgroundColor: '#fff',
    display: 'flex',
    flexDirection: 'column',
    height: '100%',
  };

  const contentStyle = {
    flexGrow: 1,
    overflowY: 'auto',
  };

  const commonResizeConfig = {
    top: true,
    right: true,
    bottom: true,
    left: true,
    topRight: true,
    bottomRight: true,
    bottomLeft: true,
    topLeft: true,
  };

  const renderBox = (key, content,id ="") => (
    <Rnd
    key={key}
      default={{
        x: boxPositions[key].x,
        y: boxPositions[key].y,
        width: boxPositions[key].width,
        height: boxPositions[key].height,
      }}
      size={{ width: boxPositions[key].width, height: boxPositions[key].height }}
      position={{ x: boxPositions[key].x, y: boxPositions[key].y }}
      minWidth={200}
      minHeight={100}
      enableResizing={!useFixedLayout}
      disableDragging={useFixedLayout}
      onDragStop={(e, d) => {
        if (!useFixedLayout) {
          setBoxPositions((prev) => ({
            ...prev,
            [key]: { ...prev[key], x: d.x, y: d.y },
          }));
        }
      }}
      onResize={(e, direction, ref, delta, position) => {
        if (!useFixedLayout) {
          setBoxPositions((prev) => ({
            ...prev,
            [key]: {
              ...prev[key],
              width: ref.offsetWidth,
              height: ref.offsetHeight,
              x: position.x,
              y: position.y,
            },
          }));
        }
      }}
      dragHandleClassName="handle"
    >
      <div ref={boxRefs[key]} id={id} style={{ ...commonBoxStyle, height: '100%' }}>
        <div
          className="handle"
          style={{
            cursor: useFixedLayout ? 'default' : 'move',
            padding: '10px',
            backgroundColor: '#e0e0e0',
            borderBottom: '1px solid #ccc',
            fontWeight: 'bold',
          }}
        >
          {key.replace(/([A-Z])/g, ' $1').trim()}
        </div>
        <div style={{ height: 'calc(100% - 40px)', overflowY: 'auto' }}>
          {content}
        </div>
      </div>
    </Rnd>
  );

  return (
    <div className="container mx-auto p-4">
      <h1 className="text-3xl font-bold mb-4">Heaven's Kitchen</h1>

      <div className="mb-4 flex items-center space-x-4" id="simulation-controls">
        <Checkbox
          id="fixed-layout"
          checked={useFixedLayout}
          onCheckedChange={handleLayoutToggle}
          label="Use Fixed Layout"
        />
        <Button onClick={saveCurrentLayout} className="flex items-center">
          <Save className="mr-2 h-4 w-4" />
          Save Current Layout
        </Button>
        <Button onClick={loadSavedLayout} className="flex items-center">
          <Download className="mr-2 h-4 w-4" />
          Load Saved Layout
        </Button>
        <Button onClick={() => setShowConfig(!showConfig)} className="flex items-center">
          <Settings className="mr-2 h-4 w-4" />
          {showConfig ? 'Hide Configuration' : 'Show Configuration'}
        </Button>
      </div>

      {renderSimulationConfig()}

      <div className="relative" style={{ height: '1000px', width: '2000px' }}>
        {renderBox(
          'moneyStatus',
          <>
            <p>Money: ${money}</p>
          </>,
        'money-status'
        )}

        {renderBox(
          'inventoryStatus',
          <>
            <ul>
              {Object.entries(inventory).map(([ing, count]) => (
                <li key={ing}>
                  {ing}: {count}
                </li>
              ))}
            </ul>
          </>,
                  'inventory-status' // ID for Joyride

        )}

        {renderBox(
          'status',
          <>
            <p>Time: {time}</p>
            <p>Groups waiting: {groups.length}</p>
            <p>Orders pending: {orders.length}</p>
            <p>Total groups served: {groupCounter - 1}</p>
            <h3 className="font-bold mt-2">Cooks:</h3>
            <ul>
              {cooks.map((cook, index) => (
                <li key={index}>
                  {cook.name}:{' '}
                  {cook.busy
                    ? `Cooking ${cook.currentOrder.name} (${cook.currentOrder.cookingTime} time units left)`
                    : 'Available'}
                </li>
              ))}
            </ul>
            <h3 className="font-bold mt-2">Incoming Deliveries:</h3>
            <ul>
              {incomingDeliveries.map((delivery, index) => (
                <li key={index}>
                  {delivery.amount} {delivery.ingredient} (Arriving in {delivery.timeLeft} time units)
                </li>
              ))}
            </ul>
          </>,

                  'status' // ID for Joyride

        )}

        {renderBox(
          'waitingGroups',
          <>
            {groups.map((group) => {
              const waitTime = time - group.arrivalTime;
              const color =
                waitTime > 60 ? 'text-red-500' : waitTime > 30 ? 'text-yellow-500' : 'text-green-500';
              return (
                <div key={group.id} className={`mb-2 ${color}`}>
                  Group {group.id} of {group.size}: Waiting for {waitTime} seconds (Completed:{' '}
                  {group.completedOrders}/{group.size})
                </div>
              );
            })}
          </>,
                  'waiting-groups' // ID for Joyride

        )}

        {renderBox('pendingOrders', renderPendingOrders(), 'pending-orders')}

        {renderBox(
          'orderIngredients',
          <>
            <Select onValueChange={(value) => setOrderIngredient(value)} value={orderIngredient}>
              <SelectTrigger className="mb-2">
                <SelectValue placeholder="Select ingredient" />
              </SelectTrigger>
              <SelectContent>
                <SelectItem value="">Select Ingredient</SelectItem>
                {INGREDIENTS.map((ing) => (
                  <SelectItem key={ing.name} value={ing.name}>
                    {ing.name} (Cost: ${ing.cost}, Delivery Time: {ing.deliveryTime})
                  </SelectItem>
                ))}
              </SelectContent>
            </Select>
            <Select
              onValueChange={(value) => setOrderAmount(Number(value))}
              value={orderAmount.toString()}
            >
              <SelectTrigger className="mb-2">
                <SelectValue placeholder="Select amount" />
              </SelectTrigger>
              <SelectContent>
                <SelectItem value="0">Select Amount</SelectItem>
                {[5, 10, 15, 20, 25].map((amount) => (
                  <SelectItem key={amount} value={amount.toString()}>
                    {amount}
                  </SelectItem>
                ))}
              </SelectContent>
            </Select>
            <Button onClick={handleOrderIngredient}>Order Ingredient</Button>
          </>,
                  'order-ingredients' // ID for Joyride

        )}

        {renderBox(
          'eventLog',
          <>
            <div>
              {logs.map((log, index) => (
                <p key={index}>{log}</p>
              ))}
            </div>
          </>,
                  'event-log' // ID for Joyride

        )}
        {renderBox(
          'alerts',
          <>
            <div>
              {alerts.map((alert) => (
                <p key={alert.id} className="text-red-500">
                  Time {alert.time}: {alert.message}
                </p>
              ))}
            </div>
          </>,
                  'alerts' // ID for Joyride

        )}
        {renderBox(
          'hireCook',
          <>
            <p>Current cooks: {cooks.length}</p>
            <p>Cost for next cook: ${cookCost}</p>
            <Button onClick={hireCook} className="mt-2">
              Hire New Cook (${cookCost})
            </Button>
          </>,
        'hire-cook' // ID for Joyride

        )}

        {renderBox(
          'menuDetails',
          <>
            {MENU_ITEMS.map((item, index) => (
              <div key={index} className="mb-4">
                <h3 className="font-bold text-lg">{item.name}</h3>
                <p>Price: ${item.price}</p>
                <p>Cooking Time: {item.cookingTime} seconds</p>
                <p className="font-semibold mt-1">Ingredients:</p>
                <ul className="list-disc list-inside">
                  {item.ingredients.map((ing, i) => (
                    <li key={i}>{ing}</li>
                  ))}
                </ul>
              </div>
            ))}
          </>,
        'menu-details' // ID for Joyride

        )}

        {renderBox(
          'kitchenGUI',
          <>
            <Card>
              <KitchenGUI cooks={cooks} inventory={inventory} tables={groups} />
            </Card>
          </>,
        'kitchen-gui' // ID for Joyride

        )}
      </div>
      <div className="mt-4 flex space-x-4">

        <Button onClick={() => setLogs([])}>
          <Trash2 className="mr-2 h-4 w-4" />
          Clear Log
        </Button>
        <Button
          onClick={() => setIsRunning(!isRunning)}
          className={`mb-4 ${!isRunning ? 'bg-green-500 hover:bg-green-600' : ''}`}
        >
          {isRunning ? (
            <Pause className="mr-2 h-4 w-4" />
          ) : (
            <Play className="mr-2 h-4 w-4" />
          )}
          {isRunning ? 'Pause Simulation' : 'Start Simulation'}
        </Button>
      </div>
    </div>
  );
};

export default RestaurantSimulation;
"""
        "Your output should only be the advices and nothing else. The advices should be on the information boxes they are not paying enough attention to. Please be consise. At most 10 to 15 words each."
    ,document_novice],
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            response_schema=list[advices]
        )
    )
    # adv_text is the raw text from the model; parse it if needed, or assume it's a list of advice strings.
    # If adv_text is a single string with multiple advices, split them.
    # Ensure `adv_data` ends up as a list of strings.
    # Update global advice data
    advice_data = json.loads(response.text)[0]['advices']
    print("Advice updated:", advice_data)
    print(f"Time taken = {time.time()-current_time}")

def periodic_advice_updater():
    while True:
        time.sleep(120)  # wait two minutes
        generate_advice()
        print("Updated advice_data every two minutes.")



# Configure your Generative AI
class advices(typing.TypedDict):
    advices: list[str]



class advices(typing.TypedDict):
    advices: list[str]

# model = genai.GenerativeModel.from_cached_content(cache)

gaze_point = None
bounding_boxes = []
connected_clients = set()  # Keep track of connected websocket clients


def process_gaze_data(input_csv, output_csv):
    try:
        df = pd.read_csv(input_csv)
        print(f"Successfully read {len(df)} rows from {input_csv}")
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return

    def convert_to_seconds(ts):
        try:
            dt = pd.to_datetime(ts)
            return dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1000000
        except Exception as e:
            print(f"Error converting timestamp {ts}: {e}")
            return 0

    df['seconds'] = df['timestamp'].apply(convert_to_seconds)
    first_timestamp = df['seconds'].iloc[0]

    current_area = None
    start_time = None
    sequences = []

    for index, row in df.iterrows():
        if pd.notna(row['bounding_box_name']):
            if current_area != row['bounding_box_name']:
                if current_area is not None:
                    sequences.append({
                        'start_time': start_time - first_timestamp,
                        'end_time': row['seconds'] - first_timestamp,
                        'area': current_area
                    })
                current_area = row['bounding_box_name']
                start_time = row['seconds']

    # Add the last sequence if exists
    if current_area is not None:
        sequences.append({
            'start_time': start_time - first_timestamp,
            'end_time': df['seconds'].iloc[-1] - first_timestamp,
            'area': current_area
        })

    formatted_sequences = []
    for seq in sequences:
        start_time_delta = str(timedelta(seconds=int(seq['start_time']))).split(':')[1:3]
        end_time_delta = str(timedelta(seconds=int(seq['end_time']))).split(':')[1:3]

        formatted_sequences.append({
            'start_time': ':'.join(start_time_delta),
            'end_time': ':'.join(end_time_delta),
            'interface_element': seq['area'],
            'duration_seconds': round(seq['end_time'] - seq['start_time'], 2)
        })

    output_df = pd.DataFrame(formatted_sequences)
    try:
        output_df.to_csv(output_csv, index=False)
        # print(f"Successfully wrote {len(output_df)} sequences to {output_csv}")
    except Exception as e:
        print(f"Error writing to CSV file: {e}")


async def handle_connection(websocket, path):
    global bounding_boxes, connected_clients
    # Add this client to the set of connected clients
    connected_clients.add(websocket)
    print("Client connected:", websocket.remote_address)
    try:
        async for message in websocket:
            elements_data = json.loads(message)
            if isinstance(elements_data, list):
                for data in elements_data:
                    xpath = data.get("xpath")
                    x = data.get("x")
                    y = data.get("y")
                    width = data.get("width")
                    height = data.get("height")
                    name = data.get("name")
                    bounding_boxes = [box for box in bounding_boxes if box['xpath'] != xpath]
                    if xpath and x is not None and y is not None and width and height:
                        bounding_boxes.append({
                            'xpath': xpath,
                            'name': name,
                            'x': int(x),
                            'y': int(y),
                            'width': int(width),
                            'height': int(height)
                        })
    except websockets.exceptions.ConnectionClosed:
        print("Client disconnected:", websocket.remote_address)
    finally:
        # Remove this client from the set of connected clients
        connected_clients.remove(websocket)


def create_output_directories():
    user_id = input("please enter the id of the user: ")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    video_dir = r"recorded_videos"
    csv_dir = r"gaze_data"
    os.makedirs(video_dir, exist_ok=True)
    os.makedirs(csv_dir, exist_ok=True)
    video_path = os.path.join(video_dir, f"screen_recording_{user_id}.mp4")
    csv_path = os.path.join(csv_dir, f"gaze_data_{user_id}.csv")
    return user_id, video_path, csv_path

def draw_cursor(frame, x, y, cursor_type):
    cv2.drawMarker(frame, (x, y), (0, 0, 0), cv2.MARKER_CROSS, 10, 2)


async def send_advice_to_clients(adv_text):
    global connected_clients
    message = json.dumps({"type": "advice", "payload": adv_text})
    await asyncio.gather(*[client.send(message) for client in connected_clients])

def tcp_server(queue_gaze):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
        server.bind(('127.0.0.1', 6554))
        server.listen(1)
        print("TCP Server listening for gaze data")
        conn, addr = server.accept()
        with conn:
            print(f"Connected by {addr}")
            while True:
                data = conn.recv(1024).decode('utf-8').strip()
                if data == "Finished":
                    print("We are done recording")
                    break
                else:
                    if data:
                        data_list = data.split(',')
                        if len(data_list) == 2:
                            try:
                                gaze_x = int(float(data_list[0]))
                                gaze_y = int(float(data_list[1]))
                                queue_gaze.put((gaze_x, gaze_y))
                            except ValueError:
                                print("Invalid gaze data received.")

In [ ]:
generate_advice()

In [ ]:
from pyibl import Agent
a = Agent(name="My Agent")

In [ ]:
a = Agent(name="My Agent")

In [ ]:
a.default_utility = 10.0

In [ ]:
b._memory = a._memory.copy()


In [ ]:
a.choose(["The Green Button", "The Red Button"])

In [ ]:
b.choose(["The Green Button", "The Red Button"])

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt

def main():
    # 1. Read CSV File
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # <-- Replace with your actual CSV file path
    df = pd.read_csv(csv_filename)  # if it's tab-separated
    # If it's comma-separated, use: df = pd.read_csv(csv_filename)

    # Quick look at the data
    print("Head of the DataFrame:")
    print(df.head(), "\n")

    print("Summary Statistics:")
    print(df.describe(), "\n")

    # 2. T-TEST ON THE EFFECT OF AI-FEEDBACK
    # --------------------------------------
    # We'll compare "Net Money" between AI-Feedback = 1 and AI-Feedback = -1.

    df_feedback_1 = df[df["AI-Feedback"] == 1]["Net Money"].dropna()
    df_feedback_neg1 = df[df["AI-Feedback"] == -1]["Net Money"].dropna()

    # Perform t-test
    t_stat, p_val = stats.ttest_ind(df_feedback_1, df_feedback_neg1, equal_var=False)
    print("T-test on Net Money by AI Feedback:")
    print(f"T-statistic = {t_stat:.3f}, p-value = {p_val:.3f}\n")

    # 3. DISPERSION EFFECT (VARIANCE COMPARISON)
    # ------------------------------------------
    # Example: Levene’s test for equality of variances by AI feedback.
    levene_stat, levene_p = stats.levene(df_feedback_1, df_feedback_neg1, center='mean')
    print("Levene’s test for Net Money by AI Feedback:")
    print(f"Levene statistic = {levene_stat:.3f}, p-value = {levene_p:.3f}\n")

    # Similarly, we can check dispersion for Gender groups (Gender = 1 vs. -1)
    df_gender_1 = df[df["Gender"] == 1]["Net Money"].dropna()
    df_gender_neg1 = df[df["Gender"] == -1]["Net Money"].dropna()

    levene_stat_gender, levene_p_gender = stats.levene(df_gender_1, df_gender_neg1, center='mean')
    print("Levene’s test for Net Money by Gender:")
    print(f"Levene statistic = {levene_stat_gender:.3f}, p-value = {levene_p_gender:.3f}\n")

    # 4. TWO-WAY ANOVA (Gender x AI Feedback on Net Money)
    # ----------------------------------------------------
    # We'll create a categorical model and run an ANOVA using statsmodels.
    # Convert Gender and AI-Feedback to categories if needed.
    df["Gender_cat"] = df["Gender"].astype("category")
    df["AI_cat"] = df["AI-Feedback"].astype("category")

    model = ols('Q("Net Money") ~ Gender_cat * AI_cat', data=df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)  # Type II ANOVA
    print("Two-Way ANOVA (Gender x AI Feedback) on Net Money:")
    print(anova_table, "\n")

    # 5. OTHER POTENTIAL ANALYSIS
    # ---------------------------

    # (A) Correlation (e.g., between 'Total Served' and 'Net Money')
    corr_val, corr_p = stats.pearsonr(df["Total Served"], df["Net Money"])
    print("Correlation between Total Served and Net Money:")
    print(f"Correlation = {corr_val:.3f}, p-value = {corr_p:.3f}\n")

    # (B) Group descriptive stats by AI feedback
    group_describe = df.groupby("AI-Feedback")["Net Money"].describe()
    print("Group stats for Net Money by AI Feedback:")
    print(group_describe, "\n")

    # 6. PLOTTING
    # -----------
    sns.set_style("whitegrid")

    # (A) Boxplot of Net Money by AI Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x="AI-Feedback", y="Net Money", data=df, palette="Set2")
    plt.title("Net Money by AI Feedback")
    plt.show()

    # (B) Boxplot of Net Money by Gender
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender", y="Net Money", data=df, palette="Set2")
    plt.title("Net Money by Gender")
    plt.show()

    # (C) Two-Factor Boxplot (Gender, AI Feedback)
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender_cat", y="Net Money", hue="AI_cat", data=df, palette="Set2")
    plt.title("Net Money by Gender and AI Feedback")
    plt.show()

    # (D) Scatter plot of Total Served vs Net Money
    plt.figure(figsize=(6,4))
    sns.scatterplot(x="Total Served", y="Net Money", hue="AI_cat", style="Gender_cat", data=df)
    plt.title("Total Served vs Net Money (Colored by AI Feedback, Shaped by Gender)")
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt

def main():
    # 1. Read CSV File
    # --------------------------------------------------
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # <-- Replace with your actual CSV file path
    # If your data is tab-delimited (as shown in your sample), specify `delimiter='\t'`.
    df = pd.read_csv(csv_filename)

    # Quick look at the data
    print("Head of the DataFrame:")
    print(df.head(), "\n")

    print("Summary Statistics (all numeric columns):")
    print(df.describe(), "\n")

    # Convert Gender and AI-Feedback to categorical for ANOVA
    df['Gender_cat'] = df['Gender'].astype('category')
    df['AI_cat'] = df['AI-Feedback'].astype('category')

    # 2. T-TESTS & DISPERSION (Total Served & Total Failed)
    # ----------------------------------------------------
    # We'll define a helper function to do:
    #   (a) T-test between AI feedback groups (1 vs. -1)
    #   (b) Levene’s test for variance between AI feedback groups
    #   (c) T-test & Levene's between Gender (1 vs. -1)

    def run_ttest_and_levene(metric_name, df):
        """
        Runs a t-test and Levene's test for the given metric (column name)
        between AI feedback groups and between gender groups.
        """
        # Split by AI feedback
        grp_ai_1 = df[df['AI-Feedback'] == 1][metric_name].dropna()
        grp_ai_neg1 = df[df['AI-Feedback'] == -1][metric_name].dropna()

        # T-test for AI feedback
        t_ai, p_ai = stats.ttest_ind(grp_ai_1, grp_ai_neg1, equal_var=False)

        # Levene's test for AI feedback
        lev_ai, lev_ai_p = stats.levene(grp_ai_1, grp_ai_neg1, center='mean')

        # Split by Gender
        grp_gender_1 = df[df['Gender'] == 1][metric_name].dropna()
        grp_gender_neg1 = df[df['Gender'] == -1][metric_name].dropna()

        # T-test for Gender
        t_gender, p_gender = stats.ttest_ind(grp_gender_1, grp_gender_neg1, equal_var=False)

        # Levene's test for Gender
        lev_gender, lev_gender_p = stats.levene(grp_gender_1, grp_gender_neg1, center='mean')

        # Print results
        print(f"========== {metric_name.upper()} ==========")
        print(f"AI Feedback Groups (1 vs. -1):")
        print(f"  T-test: t = {t_ai:.3f}, p-value = {p_ai:.3f}")
        print(f"  Levene’s test: stat = {lev_ai:.3f}, p-value = {lev_ai_p:.3f}\n")

        print(f"Gender Groups (1 vs. -1):")
        print(f"  T-test: t = {t_gender:.3f}, p-value = {p_gender:.3f}")
        print(f"  Levene’s test: stat = {lev_gender:.3f}, p-value = {lev_gender_p:.3f}\n")

    # Run T-tests & Levene's for Total Served
    run_ttest_and_levene("Total Served", df)

    # Run T-tests & Levene's for Total Failed
    run_ttest_and_levene("Total Failed", df)

    # 3. TWO-WAY ANOVA (Gender × AI Feedback)
    #    for Total Served & Total Failed
    # ----------------------------------------------------
    def run_two_way_anova(metric_name, df):
        model = ols(f'Q("{metric_name}") ~ Gender_cat * AI_cat', data=df).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        print(f"Two-Way ANOVA (Gender × AI Feedback) on {metric_name}:")
        print(anova_table, "\n")

    run_two_way_anova("Total Served", df)
    run_two_way_anova("Total Failed", df)

    # 4. DESCRIPTIVE STATS BY AI FEEDBACK
    # ----------------------------------------------------
    print("Descriptive stats (Total Served) by AI Feedback:")
    print(df.groupby("AI-Feedback")["Total Served"].describe(), "\n")

    print("Descriptive stats (Total Failed) by AI Feedback:")
    print(df.groupby("AI-Feedback")["Total Failed"].describe(), "\n")

    # 5. PLOTTING
    # ----------------------------------------------------
    sns.set_style("whitegrid")

    # (A) Boxplot of Total Served by AI Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x="AI-Feedback", y="Total Served", data=df, palette="Set2")
    plt.title("Total Served by AI Feedback")
    plt.show()

    # (B) Boxplot of Total Served by Gender
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender", y="Total Served", data=df, palette="Set2")
    plt.title("Total Served by Gender")
    plt.show()

    # (C) Two-Factor Boxplot (Gender, AI Feedback) for Total Served
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender_cat", y="Total Served", hue="AI_cat", data=df, palette="Set2")
    plt.title("Total Served by Gender and AI Feedback")
    plt.show()

    # (D) Boxplot of Total Failed by AI Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x="AI-Feedback", y="Total Failed", data=df, palette="Set2")
    plt.title("Total Failed by AI Feedback")
    plt.show()

    # (E) Boxplot of Total Failed by Gender
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender", y="Total Failed", data=df, palette="Set2")
    plt.title("Total Failed by Gender")
    plt.show()

    # (F) Two-Factor Boxplot (Gender, AI Feedback) for Total Failed
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender_cat", y="Total Failed", hue="AI_cat", data=df, palette="Set2")
    plt.title("Total Failed by Gender and AI Feedback")
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns

def main():
    # 1. Read your CSV
    # --------------------------------------------------
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # <-- Replace with your actual CSV file path
    # If your data is tab-delimited, use delimiter='\t'
    df = pd.read_csv(csv_filename)

    # 2. QQ PLOTS FOR ALL OUTPUT MEASURES
    # --------------------------------------------------
    # If you only want Total Served and Total Failed, remove 'Net Money' below
    output_measures = ['Total Served', 'Total Failed', 'Net Money']

    # Filter out any that might not exist in your dataset
    output_measures = [col for col in output_measures if col in df.columns]

    sns.set_style("whitegrid")

    # Create subplots
    fig, axs = plt.subplots(1, len(output_measures), figsize=(5 * len(output_measures), 5))

    # If there's only one measure, axs will not be an array, so handle that case
    if len(output_measures) == 1:
        axs = [axs]  # make it iterable

    # Generate a QQ plot for each measure
    for ax, measure in zip(axs, output_measures):
        sm.qqplot(df[measure].dropna(), line='45', ax=ax)
        ax.set_title(f"QQ Plot: {measure}")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

def main():
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # <-- Change to your actual CSV path
    # If your data is tab-delimited, include delimiter='\t'
    df = pd.read_csv(csv_filename)

    df["Total_Served_log"], lambda_served = stats.boxcox(df["Total Served"])

    # For net_money (must be positive)
    df["Net_Money_log"], lambda_money = stats.boxcox(df["Net Money"])

    # For total_failed, since you have zeros, you need to add a constant
    # Adding a small constant to handle zeros
    df["Total_Failed_log"], lambda_failed = stats.boxcox(df["Total Failed"] + 1)

    # # For the strictly positive columns, we can use plain np.log.
    # df["Total_Served_log"] = np.log(df["Total Served"])
    # df["Total_Failed_log"] = np.log(df["Total Failed"]+1)
    # df["Net_Money_log"]    = np.log(df["Net Money"])

    # List only the newly transformed columns we want to QQ plot.
    transformed_cols = ["Total_Served_log", "Total_Failed_log", "Net_Money_log"]
    def create_qq_plot(data, title):
        fig, ax = plt.subplots(figsize=(8, 6))
        stats.probplot(data, dist="norm", plot=ax)
        ax.set_title(title)
        plt.tight_layout()
        return fig

# Create QQ plots for transformed data

    sns.set_style("whitegrid")
    fig, axs = plt.subplots(1, len(transformed_cols), figsize=(5 * len(transformed_cols), 5))

    # If there's only one column, axs won't be an array, so handle that case:
    if len(transformed_cols) == 1:
        axs = [axs]

    for ax, col in zip(axs, transformed_cols):
      create_qq_plot(df[col], f"QQ Plot:{col}")
    # plt.show()

        # sm.qqplot(df[col].dropna(), line='45', ax=ax)
        # ax.set_title(f"QQ Plot: {col}")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt

def main():
    # =========================================================================
    # 1. LOAD THE DATA
    # =========================================================================
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # Replace with your actual CSV path/filename
    # If tab-separated, use delimiter='\t'. If comma-separated, omit the delimiter or set delimiter=','
    df = pd.read_csv(csv_filename)

    # Convert columns to categories (helpful for ANOVA)
    df["Gender_cat"] = df["Gender"].astype("category")
    df["AI_cat"] = df["AI-Feedback"].astype("category")

    # Quick check
    print("Data Head:\n", df.head(), "\n")
    print("Data Info:\n", df.info(), "\n")
    print("Summary Stats (all numeric columns):\n", df.describe(), "\n")

    # =========================================================================
    # 2. SPLIT BY SESSION
    # =========================================================================
    print("HHHHHHHHHHHHHHHHHHHHHHHEEEEEEEEEEEEEEEEEEEEEEEEEEEEERRRRRRRRRRRRRRRRRRRRRRRRREEEEEEEEEEEEEEEEEEEEEE")
    df_s1 = df[df["Session"] == 1].copy()
    df_s2 = df[df["Session"] == 2].copy()
    group1 = df_s1[df_s1["AI-Feedback"] == 1]["Total Served"].dropna()
    group2 = df_s1[df_s1["AI-Feedback"] == -1]["Total Served"].dropna()
    print("==============","/////////////////////////")
    print((df_s1[df_s1["AI-Feedback"] == 1]["Total Failed"].dropna()).mean(),(df_s1[df_s1["AI-Feedback"] == -1]["Total Failed"].dropna()).mean())
    print((df_s1[df_s1["AI-Feedback"] == 1]["Total Failed"].dropna()).std(),(df_s1[df_s1["AI-Feedback"] == -1]["Total Failed"].dropna()).std())
    print("Session 1==============","/////////////////////////")
    lev_stat, lev_p = stats.levene(group1, group2, center='mean')
    print("Levene’s test statistic:", lev_stat)
    print("Levene’s test p-value:", lev_p)

    group1 = df_s2[df_s2["AI-Feedback"] == 1]["Total Failed"].dropna()
    group2 = df_s2[df_s2["AI-Feedback"] == -1]["Total Failed"].dropna()
    print("Session 2==============","/////////////////////////")

    lev_stat, lev_p = stats.levene(group1, group2, center='mean')
    print("Levene’s test statistic:", lev_stat)
    print("Levene’s test p-value:", lev_p)


    # Quick check: how many records per session?
    print(f"Session 1 rows: {len(df_s1)}")
    print(f"Session 2 rows: {len(df_s2)}\n")

    # =========================================================================
    # 3. HELPER FUNCTION FOR T-TESTS AND TWO-WAY ANOVA
    # =========================================================================
    def run_ttest_and_anova(sub_df, metric_name):
        """
        Runs:
          - T-tests for AI-Feedback (1 vs -1) and Gender (1 vs -1)
          - Two-way ANOVA with Gender and AI-Feedback
        on the given sub_df (which is presumably filtered by Session).
        """
        print(f"=== Analyzing {metric_name} ===")

        # T-test for AI feedback (1 vs -1)
        grp_ai_1 = sub_df[sub_df["AI-Feedback"] == 1][metric_name].dropna()
        grp_ai_neg1 = sub_df[sub_df["AI-Feedback"] == -1][metric_name].dropna()

        if len(grp_ai_1) > 1 and len(grp_ai_neg1) > 1:
            t_stat_ai, p_val_ai = stats.ttest_ind(grp_ai_1, grp_ai_neg1, equal_var=False)
            print(f"  [AI Feedback] T-test: t={t_stat_ai:.3f}, p={p_val_ai:.3f}")
        else:
            print("  [AI Feedback] T-test: Not enough data in one or both groups.")

        # T-test for Gender (1 vs -1)
        grp_gender_1 = sub_df[sub_df["Gender"] == 1][metric_name].dropna()
        grp_gender_neg1 = sub_df[sub_df["Gender"] == -1][metric_name].dropna()

        if len(grp_gender_1) > 1 and len(grp_gender_neg1) > 1:
            t_stat_gen, p_val_gen = stats.ttest_ind(grp_gender_1, grp_gender_neg1, equal_var=False)
            print(f"  [Gender] T-test:     t={t_stat_gen:.3f}, p={p_val_gen:.3f}")
        else:
            print("  [Gender] T-test: Not enough data in one or both groups.")

        # Two-way ANOVA: Gender * AI
        # Make sure we have more than one group in each factor:
        if sub_df["Gender_cat"].nunique() > 1 and sub_df["AI_cat"].nunique() > 1:
            model = ols(f'Q("{metric_name}") ~ Gender_cat * AI_cat', data=sub_df).fit()
            anova_table = sm.stats.anova_lm(model, typ=2)
            print("  [Two-way ANOVA]:\n", anova_table, "\n")
        else:
            print("  [Two-way ANOVA]: Not enough variation in factors for ANOVA.\n")

    # =========================================================================
    # 4. (a) SESSION 1 ANALYSIS
    # =========================================================================
    print("============================================================")
    print("SESSION 1 ANALYSIS")
    print("============================================================\n")
    metrics = ["Total Served", "Total Failed"]  # Add more if needed
    for m in metrics:
        run_ttest_and_anova(df_s1, m)

    # =========================================================================
    # 5. (b) SESSION 2 ANALYSIS
    # =========================================================================
    print("============================================================")
    print("SESSION 2 ANALYSIS")
    print("============================================================\n")
    for m in metrics:
        run_ttest_and_anova(df_s2, m)

    # =========================================================================
    # 6. (c) LEARNING / INCREASE FROM SESSION 1 TO SESSION 2
    # =========================================================================
    print("============================================================")
    print("LEARNING (SESSION 2 - SESSION 1) AT THE GROUP LEVEL")
    print("============================================================\n")

    # Because we do NOT have a participant ID, we'll do a group-level comparison:
    # (Gender, AI-Feedback) x (mean of Session 1) vs. (mean of Session 2)
    # Then difference = session2_mean - session1_mean.

    for m in metrics:
        print(f"--- {m.upper()} ---")
        # Group by (Gender, AI-Feedback, Session), then compute mean
        group_means = df.groupby(["Gender", "AI-Feedback", "Session"])[m].mean().reset_index()

        # Pivot so that columns become Session=1 and Session=2
        pivot_means = group_means.pivot_table(
            index=["Gender","AI-Feedback"],
            columns="Session",
            values=m
        )

        # Rename the pivot columns for clarity
        pivot_means.rename(columns={1: f"{m}_S1", 2: f"{m}_S2"}, inplace=True)
        
        # Compute difference column
        pivot_means[f"{m}_diff"] = pivot_means[f"{m}_S2"] - pivot_means[f"{m}_S1"]

        # Print out the table
        print(pivot_means, "\n")

    # =========================================================================
    # 7. PLOTTING
    # =========================================================================
    sns.set_style("whitegrid")
    df_s1.rename(columns = {"Total Served": "Groups Served"},inplace= True)
    df_s2.rename(columns = {"Total Served": "Groups Served"},inplace= True)

    # (a) For each session, boxplots of Total Served by AI feedback
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df_s1, x="AI-Feedback", y="Groups Served", palette="Set2")
    plt.xticks([0, 1], ["Without AI Feedback", "With AI Feedback"])
    # plt.title("Session 1: Total Served by AI Feedback")

    plt.show()

    plt.figure(figsize=(8,4))
    sns.boxplot(data=df_s2, x="AI-Feedback", y="Groups Served", palette="Set2")
    # plt.title("Session 2: Total Served by AI Feedback")
    plt.xticks([0, 1], ["Without AI Feedback", "With AI Feedback"])
    plt.show()

    # (b) For each session, boxplots of Total Failed by Gender
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df_s1, x="Gender", y="Total Failed", palette="Set2")
    plt.title("Session 1: Total Failed by Gender")
    plt.show()

    plt.figure(figsize=(8,4))
    sns.boxplot(data=df_s2, x="Gender", y="Total Failed", palette="Set2")
    plt.title("Session 2: Total Failed by Gender")
    plt.show()

    # (c) Optionally, you can do side-by-side comparisons or other visualizations.
    # For instance, a bar chart showing the group-level means by Session:
    # We'll do an example for "Total Served" by (Gender, AI-Feedback).

    # Prepare data for a grouped bar plot
    group_means_served = df.groupby(["Gender","AI-Feedback","Session"])["Total Served"].mean().reset_index()
    group_means_served.rename(columns = {"Total Served": "Groups Served"},inplace= True)
    plt.figure(figsize=(8,5))
    sns.barplot(
        data=group_means_served,
        x="Session",
        y="Groups Served",
        hue="AI-Feedback",
        palette="Set2"
    )
    plt.title("Mean Total Served by Session & AI-Feedback (Grouped)")
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

def main():
    # =========================================================================
    # 1. LOAD THE DATA
    # =========================================================================
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # <-- Replace with your actual CSV file path
    # If tab-delimited, use `delimiter='\t'`.
    df = pd.read_csv(csv_filename,)

    # Quick look
    print("Data Head:\n", df.head(), "\n")
    print("Data Info:\n", df.info(), "\n")

    # =========================================================================
    # 2. SEPARATE SESSION 1 AND SESSION 2
    #    (Assuming row i in Session 1 corresponds to row i in Session 2)
    # =========================================================================
    df_s1 = df[df['Session'] == 1].copy().reset_index(drop=True)
    df_s2 = df[df['Session'] == 2].copy().reset_index(drop=True)

    # Check that we have the same number of rows in each session
    # for a proper 1-to-1 match
    print(f"Session 1 Rows: {len(df_s1)}")
    print(f"Session 2 Rows: {len(df_s2)}\n")

    # If they differ, you cannot do a perfect pairing by index.
    # You would need a participant ID or some matching logic.

    # =========================================================================
    # 3. CREATE A 'LEARNING' (DIFFERENCE) DATAFRAME
    #    For each participant (row), difference = Session1 - Session2
    # =========================================================================
    # We'll do this for "Total Served" and "Total Failed".
    # You can add other measures similarly if needed.

    # We assume that columns like Gender, AI-Feedback do NOT change across sessions
    # for the same participant. If they do, you must decide which session's values to use.
    # Here we'll just take them from Session 1's row.

    diff_df = pd.DataFrame()

    # Copy relevant grouping columns from Session 1
    diff_df['Gender'] = df_s1['Gender'].values
    diff_df['AI-Feedback'] = df_s1['AI-Feedback'].values

    # Create difference columns
    diff_df['Learning_Served'] = df_s1['Total Served'] - df_s2['Total Served']
    diff_df['Learning_Failed'] = df_s1['Total Failed'] - df_s2['Total Failed']

    # Quick peek
    print("Head of the difference DataFrame:\n", diff_df.head(), "\n")

    # Convert to categorical for ANOVA
    diff_df["Gender_cat"] = diff_df["Gender"].astype("category")
    diff_df["AI_cat"] = diff_df["AI-Feedback"].astype("category")

    # =========================================================================
    # 4. ANOVA ON THE DIFFERENCE (LEARNING MEASURE)
    # =========================================================================
    # We'll do a two-way ANOVA: difference ~ Gender * AI-Feedback

    # Example for "Learning_Served"
    model_served = ols('Learning_Served ~ Gender_cat * AI_cat', data=diff_df).fit()
    anova_served = sm.stats.anova_lm(model_served, typ=2)
    print("ANOVA on (Session1 - Session2) for Total Served:")
    print(anova_served, "\n")

    # Example for "Learning_Failed"
    model_failed = ols('Learning_Failed ~ Gender_cat * AI_cat', data=diff_df).fit()
    anova_failed = sm.stats.anova_lm(model_failed, typ=2)
    print("ANOVA on (Session1 - Session2) for Total Failed:")
    print(anova_failed, "\n")

    # =========================================================================
    # 5. ADDITIONAL TESTS OR DESCRIPTIVE STATS
    # =========================================================================
    # For instance, we can do a quick t-test on Learning_Served by AI-Feedback
    # (1 vs. -1), ignoring Gender:
    grp_ai1 = diff_df[diff_df["AI-Feedback"] == 1]["Learning_Served"]
    grp_ai_neg1 = diff_df[diff_df["AI-Feedback"] == -1]["Learning_Served"]

    t_stat, p_val = stats.ttest_ind(grp_ai1, grp_ai_neg1, equal_var=False)
    print("T-test on Learning_Served by AI Feedback (1 vs. -1):")
    print(f"T-stat={t_stat:.3f}, p-value={p_val:.3f}\n")

    # =========================================================================
    # 6. PLOT LEARNING DISTRIBUTIONS
    # =========================================================================
    sns.set_style("whitegrid")

    # Boxplot: Learning_Served by AI Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x="AI-Feedback", y="Learning_Served", data=diff_df, palette="Set2")
    plt.title("Learning (S1 - S2) in Total Served by AI Feedback")
    plt.show()

    # Boxplot: Learning_Served by Gender
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender", y="Learning_Served", data=diff_df, palette="Set2")
    plt.title("Learning (S1 - S2) in Total Served by Gender")
    plt.show()

    # Boxplot: Learning_Failed by AI Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x="AI-Feedback", y="Learning_Failed", data=diff_df, palette="Set2")
    plt.title("Learning (S1 - S2) in Total Failed by AI Feedback")
    plt.show()

    # Boxplot: Learning_Failed by Gender
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Gender", y="Learning_Failed", data=diff_df, palette="Set2")
    plt.title("Learning (S1 - S2) in Total Failed by Gender")
    plt.show()



if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_ai_feedback_sessions(df, performance_col="Total Served"):
    """
    Creates a grouped bar plot showing the mean performance_col
    in Session 1 and Session 2 for AI-Feedback = 1 vs. -1.
    """
    df['Session'] = df['Session'].astype(str)
    # Group by AI-Feedback and Session, computing the mean performance
    grouped_df = (
        df.groupby(["AI-Feedback", "Session"])[performance_col]
          .mean()
          .reset_index(name="mean_performance")
    )

    # Create the grouped bar plot
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=grouped_df,
        x="AI-Feedback",
        y="mean_performance",
        hue="Session",
        palette="Set2"
    )

    # Add labels and title
    plt.xlabel("AI Feedback (1=With Feedback, -1=No Feedback)")
    plt.ylabel(f"Mean {performance_col}")
    plt.title(f"{performance_col} by AI Feedback and Session")
    plt.legend(title="Session")
    plt.tight_layout()
    plt.show()

# Example usage
if __name__ == "__main__":
    # 1. Read your CSV file (assuming tab-delimited as per your data snippet)
    df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

    # 2. Plot bar chart
    plot_ai_feedback_sessions(df, performance_col="Total Served")

    # Optionally, for "Total Failed" or another column:
    # plot_ai_feedback_sessions(df, performance_col="Total Failed")


In [ ]:
df.loc[1]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_ai_feedback_sessions_with_variation(df, performance_col="Total Served"):
    """
    Creates a grouped bar plot of mean performance_col for Session 1 and Session 2
    at AI Feedback values (1 vs -1), with error bars indicating ±1 SD.
    """
    plt.figure(figsize=(8, 5))
    df['Session'] = df['Session'].astype(str)
    # Use Seaborn's barplot with raw data.
    # - It will automatically group by x and hue, compute mean, and show ±1 SD (ci='sd').
    # - 'capsize' adds whiskers at the end of error bars.
    sns.barplot(
        data=df,
        x="AI-Feedback",
        y=performance_col,
        hue="Session",
        palette="Set2",
        errorbar='sd',        # or ci=68 for a 68% CI, ci=95 for a 95% CI, etc.
        capsize=0.05
    )

    plt.xlabel("")
    plt.ylabel(f"Mean {performance_col} ± 1 SD")
    plt.xticks([0, 1], ["Without AI Feedback", "With AI Feedback"])
    # plt.title(f"{performance_col} by AI Feedback and Session (with Variation)")
    plt.legend(title="Session", loc="best")
    plt.tight_layout()
    plt.show()

# Example usage
if __name__ == "__main__":
    # 1. Read your CSV file (tab-delimited, based on your data snippet)
    df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

    # 2. Plot bar chart with variation (here for "Total Served")
    plot_ai_feedback_sessions_with_variation(df, performance_col="Total Served")

    # If you want to plot for a different measure (e.g., "Total Failed"), just change:
    # plot_ai_feedback_sessions_with_variation(df, performance_col="Total Failed")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_ai_feedback_sessions_with_variation(df, performance_col="Total Served"):
    """
    Creates a grouped bar plot of mean performance_col for Session 1 and Session 2
    at AI Feedback values (1 vs -1), with error bars indicating ±1 SD.
    Also adds mode and number of users in mode as text on the plot.
    """
    plt.figure(figsize=(8, 5))

    # Calculate mode and count for each group
    groups = df.groupby(["AI-Feedback", "Session"])[performance_col]
    mode_values = groups.agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan)  # Handle empty modes
    mode_counts = groups.agg(lambda x: x.value_counts().max() if len(x) > 0 else 0)     # Handle empty groups
    df['Session'] = df['Session'].astype(str)
    # Create bar plot (same as before)
    sns.barplot(
        data=df,
        x="AI-Feedback",
        y=performance_col,
        hue="Session",
        palette="Set2",
        ci="sd",
        capsize=0.2
    )

    # Add mode and count as text annotations
    for (ai_feedback, session), mode in mode_values.items():
        count = mode_counts.loc[(ai_feedback, session)]
        x_pos = (ai_feedback + (session - 1) * 0.4)  # Adjust x position for grouping
        y_pos = mode + 1  # Adjust y position as needed
        plt.text(x_pos, y_pos, f"Mode: {mode} (n={count})", ha="center", va="bottom", fontsize=8)

    # Rest of the plot configuration (same as before)
    plt.xlabel("")
    plt.ylabel(f"Mean {performance_col} ± 1 SD")
    plt.xticks([0, 1], ["No AI Feedback", "With AI Feedback"])
    plt.legend(title="Session", loc="best")
    plt.tight_layout()
    plt.show()

In [ ]:
df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

# 2. Plot bar chart with variation (here for "Total Served")
plot_ai_feedback_sessions_with_variation(df, performance_col="Total Served")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_violin_ai_feedback_sessions(df, performance_col="Total Served"):
    """
    Creates a violin plot of the distribution of performance_col
    grouped by AI-Feedback (x-axis) and split by Session (hue).

    :param df: pandas DataFrame with columns: 'AI-Feedback', 'Session', performance_col
    :param performance_col: name of the numeric column to plot (default='Total Served')
    """
    plt.figure(figsize=(8,5))

    # Violin plot with distributions split by 'Session' at each AI-Feedback level
    sns.violinplot(
        data=df,
        x="AI-Feedback",
        y=performance_col,
        hue="Session",
        palette="Set2",
        split=True,       # Combines both sessions into a single violin per AI-Feedback category
        inner="quartile"  # Shows quartiles inside the violin. You can use 'box', 'point', or None.
    )
    plt.xticks([0, 1], ["No AI Feedback", "With AI Feedback"])
    # Customize labels and title
    plt.xlabel("AI Feedback")
    plt.ylabel(performance_col)
    # plt.title(f"Distribution of {performance_col} by AI Feedback and Session")
    plt.legend(title="Session", loc="best")
    plt.tight_layout()
    plt.show()

# =============================================================================
# Example usage:
# =============================================================================
if __name__ == "__main__":
    # 1. Read your CSV file (adjust delimiter as needed)
    df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

    # 2. Plot violin chart for "Total Served"
    plot_violin_ai_feedback_sessions(df, performance_col="Total Served")

    # 3. If you also want to plot "Total Failed," simply call:
    # plot_violin_ai_feedback_sessions(df, performance_col="Total Failed")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_violin_session_feedback(df, performance_col="Total Served"):
    """
    Creates a violin plot of performance_col by Session,
    and splits each violin by AI-Feedback (1 = with feedback, -1 = no feedback).

    Args:
        df (pd.DataFrame): Your dataset with columns 'Session', 'AI-Feedback', and performance_col.
        performance_col (str): The numeric column you want to plot (default='Total Served').
    """
    plt.figure(figsize=(8, 5))

    # Create the violin plot
    sns.violinplot(
        data=df,
        x="Session",               # Put Session on the x-axis
        y=performance_col,         # The numeric performance measure
        hue="AI-Feedback",         # Split the violin by AI Feedback
        palette="Set2",
        split=True,                # Splits the violins into two halves (1 vs. -1)
        inner="box"                # Draw a miniature boxplot inside each violin
        # Other options for 'inner': 'point', 'quartile', or None
    )
    plt.xticks([0, 1], ["Session 1", "Session 2"])
    # Labeling
    plt.xlabel("Session")
    plt.ylabel(performance_col)
    plt.title(f"{performance_col} by Session, split by AI Feedback")
    plt.legend(title="AI Feedback", loc="best")
    plt.tight_layout()
    plt.show()

# Example usage
if __name__ == "__main__":
    # 1. Read your CSV file (adjust delimiter if needed)
    df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

    # 2. Generate the violin plot for "Total Served"
    plot_violin_session_feedback(df, performance_col="Total Served")

    # 3. If you want to plot another metric, just call the function with a different performance_col:
    plot_violin_session_feedback(df, performance_col="Total Failed")


In [ ]:
pip install pingouin


In [ ]:
import pandas as pd
import pingouin as pg  # pip install pingouin
import seaborn as sns
import matplotlib.pyplot as plt

def main():
    """
    Script to compute (Session 1 - Session 2) as 'learning' and perform
    Welch’s ANOVA on that difference by AI Feedback (only).
    """

    # ----------------------------------------------------------------------
    # 1) Load the data
    # ----------------------------------------------------------------------
    csv_filename = "../../data/synthetic/Participants_SYNTHETIC.csv"  # <-- Replace with your actual file/path
    # If your file is tab-delimited, specify delimiter='\t'
    df = pd.read_csv(csv_filename)

    print("Data Head:\n", df.head(), "\n")
    print("Data Info:")
    print(df.info(), "\n")

    # ----------------------------------------------------------------------
    # 2) Separate rows for Session 1 and Session 2
    #    (Assumes row i in Session 1 matches row i in Session 2)
    # ----------------------------------------------------------------------
    df_s1 = df[df['Session'] == 1].copy().reset_index(drop=True)
    df_s2 = df[df['Session'] == 2].copy().reset_index(drop=True)

    print(f"Number of rows in Session 1: {len(df_s1)}")
    print(f"Number of rows in Session 2: {len(df_s2)}\n")

    if len(df_s1) != len(df_s2):
        print("WARNING: Session 1 and Session 2 do NOT have the same number of rows!")
        print("If this is repeated measures, you need a ParticipantID to merge properly.\n")

    # ----------------------------------------------------------------------
    # 3) Compute 'learning' measures (Session 1 - Session 2)
    # ----------------------------------------------------------------------
    # We'll do it for 'Total Served' and 'Total Failed' as examples.
    diff_df = pd.DataFrame()

    # Keep AI-Feedback from Session 1 (assuming it doesn't change across sessions)
    diff_df['AI-Feedback'] = df_s1['AI-Feedback'].values

    diff_df['Learning_Served'] = df_s1['Total Served'] - df_s2['Total Served']
    diff_df['Learning_Failed'] = df_s1['Total Failed'] - df_s2['Total Failed']

    print("Head of the difference DataFrame:\n", diff_df.head(), "\n")

    # ----------------------------------------------------------------------
    # 4) Welch’s ANOVA on Learning_Served
    #    (AI Feedback as the single factor)
    # ----------------------------------------------------------------------
    print("====== Welch’s ANOVA on Learning_Served by AI Feedback ======")

    # pingouin welch_anova => one-way Welch’s ANOVA
    welch_result_served = pg.welch_anova(dv='Learning_Served', between='AI-Feedback', data=diff_df)
    print(welch_result_served, "\n")

    # If AI-Feedback truly has only 2 levels, a post-hoc test is basically
    # the same as a Welch’s t-test. But here's how you'd do Games-Howell anyway:
    print("------ Games-Howell post-hoc (Learning_Served) ------")
    games_howell_served = pg.pairwise_gameshowell(dv='Learning_Served', between='AI-Feedback', data=diff_df)
    print(games_howell_served, "\n")

    # ----------------------------------------------------------------------
    # 5) Welch’s ANOVA on Learning_Failed
    # ----------------------------------------------------------------------
    print("====== Welch’s ANOVA on Learning_Failed by AI Feedback ======")
    welch_result_failed = pg.welch_anova(dv='Learning_Failed', between='AI-Feedback', data=diff_df)
    print(welch_result_failed, "\n")

    print("------ Games-Howell post-hoc (Learning_Failed) ------")
    games_howell_failed = pg.pairwise_gameshowell(dv='Learning_Failed', between='AI-Feedback', data=diff_df)
    print(games_howell_failed, "\n")

    # ----------------------------------------------------------------------
    # 6) Plot the results
    # ----------------------------------------------------------------------
    sns.set_style("whitegrid")

    # Boxplot: Learning_Served by AI-Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x='AI-Feedback', y='Learning_Served', data=diff_df, palette="Set2")
    plt.title("Learning (S1 - S2) in Total Served by AI Feedback")
    plt.show()

    # Boxplot: Learning_Failed by AI-Feedback
    plt.figure(figsize=(6,4))
    sns.boxplot(x='AI-Feedback', y='Learning_Failed', data=diff_df, palette="Set2")
    plt.title("Learning (S1 - S2) in Total Failed by AI Feedback")
    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_violin_ai_feedback_sessions(df, performance_col="Total Served"):
    """
    Creates a violin plot of the distribution of performance_col
    grouped by AI-Feedback (x-axis) and split by Session (hue).

    :param df: pandas DataFrame with columns: 'AI-Feedback', 'Session', performance_col
    :param performance_col: name of the numeric column to plot (default='Total Served')
    """
    plt.figure(figsize=(8,5))

    # Violin plot with distributions split by 'Session' at each AI-Feedback level
    sns.violinplot(
        data=df,
        x="AI-Feedback",
        y=performance_col,
        hue="Session",
        palette="Set2",
        split=True,       # Combines both sessions into a single violin per AI-Feedback category
        inner="box"       # Changed from "quartile" to "box" to include whiskers (min and max)
    )
    plt.xticks([0, 1], ["No AI Feedback", "With AI Feedback"])
    # Customize labels and title
    plt.xlabel("AI Feedback")
    plt.ylabel(performance_col)
    # plt.title(f"Distribution of {performance_col} by AI Feedback and Session")
    plt.legend(title="Session", loc="best")
    plt.tight_layout()
    plt.show()

# =============================================================================
# Example usage:
# =============================================================================
if __name__ == "__main__":
    # 1. Read your CSV file (adjust delimiter as needed)
    df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

    # 2. Plot violin chart for "Total Served"
    plot_violin_ai_feedback_sessions(df, performance_col="Total Served")

    # 3. If you also want to plot "Total Failed," simply call:
    # plot_violin_ai_feedback_sessions(df, performance_col="Total Failed")


In [ ]:
# ================================
# Requirements:
#   pip install pingouin pandas scipy statsmodels scikit-posthocs
# ================================

import pandas as pd
import numpy as np
from scipy.stats import shapiro, levene, kruskal
import pingouin as pg
from scipy import stats

# ---- Load & prep ----
df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

# Standardize column names (no spaces) for modeling
df = df.rename(columns={
    "Participant ID": "ParticipantID",
    "AI-Feedback": "AI_Feedback",
    "Total Served": "Total_Served",
    "Total Failed": "Total_Failed",
    "Money Left": "Money_Left",
    "Net Money": "Net_Money"
})

# Ensure factors are categorical
# (Assumes Gender and AI_Feedback are coded as 1 / -1)
df["Gender"] = df["Gender"].map({1: "Male", -1: "Female"}).astype("category")
df["AI_Feedback"] = df["AI_Feedback"].map({1: "AI_On", -1: "AI_Off"}).astype("category")
df["Session"] = df["Session"].astype(int)

# Split by phase
df_train = df[df["Session"] == 1].copy()
df_test  = df[df["Session"] == 2].copy()

print("=== DATA OVERVIEW ===")
print(f"Training phase: {len(df_train)} participants")
print(f"Testing phase: {len(df_test)} participants")
print("\nGroup sizes (Training):")
print(df_train.groupby(['Gender', 'AI_Feedback']).size())
print("\nGroup sizes (Testing):")
print(df_test.groupby(['Gender', 'AI_Feedback']).size())

# -------------------------
# 1) CORRECTED Normality Testing
# -------------------------
def comprehensive_normality_check(data, dv="Total_Served", phase_name=""):
    """
    Comprehensive normality check including:
    - Shapiro-Wilk test for each group
    - Overall normality assessment
    - Homogeneity of variance test
    """
    print(f"\n=== NORMALITY CHECK: {phase_name} ===")
    
    # Check normality for each group
    normality_results = []
    groups_data = []
    
    for gender in data["Gender"].cat.categories:
        for ai_feedback in data["AI_Feedback"].cat.categories:
            group_data = data[(data["Gender"] == gender) & (data["AI_Feedback"] == ai_feedback)][dv].dropna()
            
            if len(group_data) >= 3:
                W, p_shapiro = shapiro(group_data)
                normality_results.append({
                    "Gender": gender, 
                    "AI_Feedback": ai_feedback, 
                    "N": len(group_data),
                    "Mean": group_data.mean(),
                    "Std": group_data.std(),
                    "Shapiro-W": W, 
                    "Shapiro-p": p_shapiro,
                    "Normal": "Yes" if p_shapiro > 0.05 else "No"
                })
                groups_data.append(group_data)
            else:
                normality_results.append({
                    "Gender": gender, 
                    "AI_Feedback": ai_feedback, 
                    "N": len(group_data),
                    "Mean": np.nan,
                    "Std": np.nan,
                    "Shapiro-W": np.nan, 
                    "Shapiro-p": np.nan,
                    "Normal": "N/A"
                })
    
    norm_df = pd.DataFrame(normality_results)
    print(norm_df.to_string(index=False))
    
    # Overall assessment
    valid_tests = norm_df.dropna(subset=["Shapiro-p"])
    all_normal = (valid_tests["Shapiro-p"] > 0.05).all()
    print(f"\nAll groups pass normality test: {all_normal}")
    
    # Homogeneity of variance test (Levene's test)
    if len(groups_data) >= 2 and all(len(g) >= 2 for g in groups_data):
        levene_stat, levene_p = levene(*groups_data)
        print(f"Levene's test for equal variances: F={levene_stat:.4f}, p={levene_p:.4f}")
        equal_variances = levene_p > 0.05
        print(f"Equal variances assumption met: {equal_variances}")
        
        # Overall decision
        parametric_ok = all_normal and equal_variances
        print(f"\nParametric ANOVA appropriate: {parametric_ok}")
        return parametric_ok, norm_df
    else:
        print("Cannot perform Levene's test (insufficient data)")
        return False, norm_df

# Check normality for both phases
param_ok_train, norm_train = comprehensive_normality_check(df_train, "Total_Served", "Training")
param_ok_test, norm_test = comprehensive_normality_check(df_test, "Total_Served", "Testing")

# ================================================================
# 2) ALIGNED RANK TRANSFORM (ART) TEST
# ================================================================

def aligned_rank_transform_anova(data, dv, factor1, factor2):
    """
    Perform Aligned Rank Transform ANOVA for two-way factorial design.
    This is the proper non-parametric equivalent of two-way ANOVA.
    """
    print(f"\n=== ALIGNED RANK TRANSFORM ANOVA ===")
    print("This test allows for proper analysis of main effects and interactions")
    print("in a non-parametric two-way factorial design.\n")
    
    # Create a working copy and ensure DV is numeric
    df_work = data.copy()
    df_work[dv] = pd.to_numeric(df_work[dv], errors='coerce')
    
    # Step 1: Calculate marginal means for alignment
    # Overall mean
    grand_mean = df_work[dv].mean()
    
    # Main effect means
    factor1_means = df_work.groupby(factor1)[dv].mean()
    factor2_means = df_work.groupby(factor2)[dv].mean()
    
    # Cell means for interaction
    cell_means = df_work.groupby([factor1, factor2])[dv].mean()
    
    # Step 2: Create aligned scores for each effect
    # Convert categorical columns to string for mapping
    df_work[f'{factor1}_str'] = df_work[factor1].astype(str)
    df_work[f'{factor2}_str'] = df_work[factor2].astype(str)
    
    # Create mapping dictionaries with string keys
    factor1_means_dict = {str(k): v for k, v in factor1_means.items()}
    factor2_means_dict = {str(k): v for k, v in factor2_means.items()}
    cell_means_dict = {(str(k[0]), str(k[1])): v for k, v in cell_means.items()}
    
    # For main effect of factor1 (remove factor2 effect)
    df_work['factor2_mean'] = df_work[f'{factor2}_str'].map(factor2_means_dict)
    df_work['aligned_factor1'] = df_work[dv] - df_work['factor2_mean'] - grand_mean
    df_work['rank_factor1'] = df_work['aligned_factor1'].rank()
    
    # For main effect of factor2 (remove factor1 effect)
    df_work['factor1_mean'] = df_work[f'{factor1}_str'].map(factor1_means_dict)
    df_work['aligned_factor2'] = df_work[dv] - df_work['factor1_mean'] - grand_mean
    df_work['rank_factor2'] = df_work['aligned_factor2'].rank()
    
    # For interaction effect (remove both main effects)
    df_work['aligned_interaction'] = (df_work[dv] - 
                                    df_work['factor1_mean'] - 
                                    df_work['factor2_mean'] + 
                                    grand_mean)
    df_work['rank_interaction'] = df_work['aligned_interaction'].rank()
    
    # Step 3: Run ANOVA on ranked aligned scores
    print("Testing Main Effect of", factor1)
    anova_factor1 = pg.anova(dv='rank_factor1', between=[factor1], data=df_work)
    print(anova_factor1.to_string(index=False))
    
    print(f"\nTesting Main Effect of {factor2}")
    anova_factor2 = pg.anova(dv='rank_factor2', between=[factor2], data=df_work)
    print(anova_factor2.to_string(index=False))
    
    print(f"\nTesting Interaction Effect ({factor1} × {factor2})")
    anova_interaction = pg.anova(dv='rank_interaction', between=[factor1, factor2], data=df_work)
    # Get only the interaction row
    interaction_row = anova_interaction[anova_interaction['Source'] == f'{factor1} * {factor2}']
    print(interaction_row.to_string(index=False))
    
    return {
        'main_effect_1': anova_factor1,
        'main_effect_2': anova_factor2, 
        'interaction': interaction_row,
        'data_with_ranks': df_work
    }

# ================================================================
# 3) RUN ART ANALYSIS
# ================================================================

print("\n" + "="*60)
print("RUNNING ART ANALYSIS FOR TRAINING PHASE")
print("="*60)
art_train = aligned_rank_transform_anova(df_train, 'Total_Served', 'Gender', 'AI_Feedback')

print("\n" + "="*60)
print("RUNNING ART ANALYSIS FOR TESTING PHASE") 
print("="*60)
art_test = aligned_rank_transform_anova(df_test, 'Total_Served', 'Gender', 'AI_Feedback')

# ================================================================
# 4) POST-HOC ANALYSIS (if significant effects found)
# ================================================================

def art_posthoc_analysis(data, dv, factor1, factor2, art_results):
    """
    Perform post-hoc analysis for significant ART results
    """
    print(f"\n=== POST-HOC ANALYSIS ===")
    
    # Check if main effects or interaction are significant
    main1_sig = art_results['main_effect_1']['p-unc'].iloc[0] < 0.05
    main2_sig = art_results['main_effect_2']['p-unc'].iloc[0] < 0.05
    int_sig = art_results['interaction']['p-unc'].iloc[0] < 0.05 if len(art_results['interaction']) > 0 else False
    
    print(f"Significant main effect of {factor1}: {main1_sig}")
    print(f"Significant main effect of {factor2}: {main2_sig}")  
    print(f"Significant interaction: {int_sig}")
    
    if int_sig:
        print(f"\n--- Simple Effects Analysis (due to significant interaction) ---")
        # Test effect of factor2 within each level of factor1
        for level in data[factor1].cat.categories:
            subset = data[data[factor1] == level]
            if len(subset) > 0 and subset[factor2].nunique() > 1:
                group1 = subset[subset[factor2] == subset[factor2].cat.categories[0]][dv]
                group2 = subset[subset[factor2] == subset[factor2].cat.categories[1]][dv] 
                if len(group1) > 0 and len(group2) > 0:
                    u_stat, p_val = stats.mannwhitneyu(group1, group2, alternative='two-sided')
                    print(f"{factor2} effect within {factor1}={level}: U={u_stat:.2f}, p={p_val:.4f}")
    
    elif main2_sig and not int_sig:
        print(f"\n--- Main Effect Analysis for {factor2} ---")
        # Simple Mann-Whitney for the main effect
        levels = data[factor2].cat.categories
        if len(levels) == 2:
            group1 = data[data[factor2] == levels[0]][dv]
            group2 = data[data[factor2] == levels[1]][dv]
            u_stat, p_val = stats.mannwhitneyu(group1, group2, alternative='two-sided')
            print(f"{levels[0]} vs {levels[1]}: U={u_stat:.2f}, p={p_val:.4f}")
            
            # Effect size (r = Z / sqrt(N))
            z_score = stats.norm.ppf(p_val/2) * -1 if p_val < 1 else 0
            r = abs(z_score) / np.sqrt(len(data))
            print(f"Effect size (r): {r:.3f}")

# Run post-hoc for both phases
print("\n" + "="*50)
print("POST-HOC ANALYSIS - TRAINING PHASE")
print("="*50)
art_posthoc_analysis(df_train, 'Total_Served', 'Gender', 'AI_Feedback', art_train)

print("\n" + "="*50)
print("POST-HOC ANALYSIS - TESTING PHASE")
print("="*50)
art_posthoc_analysis(df_test, 'Total_Served', 'Gender', 'AI_Feedback', art_test)

# ================================================================
# 5) SUMMARY AND INTERPRETATION
# ================================================================

print("\n" + "="*60)
print("SUMMARY AND INTERPRETATION")
print("="*60)

def interpret_results(phase_name, art_results):
    print(f"\n--- {phase_name} Phase Results ---")
    
    # Extract p-values
    gender_p = art_results['main_effect_1']['p-unc'].iloc[0]
    ai_p = art_results['main_effect_2']['p-unc'].iloc[0]  
    int_p = art_results['interaction']['p-unc'].iloc[0] if len(art_results['interaction']) > 0 else 1.0
    
    print(f"Gender main effect: p = {gender_p:.4f} {'*' if gender_p < 0.05 else 'ns'}")
    print(f"AI Feedback main effect: p = {ai_p:.4f} {'*' if ai_p < 0.05 else 'ns'}")
    print(f"Gender × AI Feedback interaction: p = {int_p:.4f} {'*' if int_p < 0.05 else 'ns'}")
    
    # Interpretation
    if int_p < 0.05:
        print("→ The effect of AI feedback depends on gender (or vice versa)")
    elif ai_p < 0.05:
        print("→ AI feedback has a significant main effect on Total Served")
    elif gender_p < 0.05:
        print("→ Gender has a significant main effect on Total Served")
    else:
        print("→ No significant effects detected")

interpret_results("Training", art_train)
interpret_results("Testing", art_test)

print(f"\n{'*'*60}")
print("NOTE: ART (Aligned Rank Transform) is the appropriate test for your design")
print("because it properly handles non-parametric two-way factorial analysis")
print("while preserving the ability to test interactions.")
print(f"{'*'*60}")

In [ ]:
# ================================
# Requirements:
#   pip install pingouin pandas scipy statsmodels scikit-posthocs
# ================================

import pandas as pd
import numpy as np
from scipy.stats import shapiro, levene, kruskal
import pingouin as pg
from scipy import stats

# ---- Load & prep ----
df = pd.read_csv("../../data/synthetic/Participants_SYNTHETIC.csv")

# Standardize column names (no spaces) for modeling
df = df.rename(columns={
    "Participant ID": "ParticipantID",
    "AI-Feedback": "AI_Feedback",
    "Total Served": "Total_Served",
    "Total Failed": "Total_Failed",
    "Money Left": "Money_Left",
    "Net Money": "Net_Money"
})

# Ensure factors are categorical
# (Assumes Gender and AI_Feedback are coded as 1 / -1)
df["Gender"] = df["Gender"].map({1: "Male", -1: "Female"}).astype("category")
df["AI_Feedback"] = df["AI_Feedback"].map({1: "AI_On", -1: "AI_Off"}).astype("category")
df["Session"] = df["Session"].astype(int)

# Split by phase
df_train = df[df["Session"] == 1].copy()
df_test  = df[df["Session"] == 2].copy()

print("=== DATA OVERVIEW ===")
print(f"Training phase: {len(df_train)} participants")
print(f"Testing phase: {len(df_test)} participants")
print("\nGroup sizes (Training):")
print(df_train.groupby(['Gender', 'AI_Feedback']).size())
print("\nGroup sizes (Testing):")
print(df_test.groupby(['Gender', 'AI_Feedback']).size())

# -------------------------
# 1) CORRECTED Normality Testing
# -------------------------
def comprehensive_normality_check(data, dv="Total_Served", phase_name=""):
    """
    Comprehensive normality check including:
    - Shapiro-Wilk test for each group
    - Overall normality assessment
    - Homogeneity of variance test
    """
    print(f"\n=== NORMALITY CHECK: {phase_name} ===")
    
    # Check normality for each group
    normality_results = []
    groups_data = []
    
    for gender in data["Gender"].cat.categories:
        for ai_feedback in data["AI_Feedback"].cat.categories:
            group_data = data[(data["Gender"] == gender) & (data["AI_Feedback"] == ai_feedback)][dv].dropna()
            
            if len(group_data) >= 3:
                W, p_shapiro = shapiro(group_data)
                normality_results.append({
                    "Gender": gender, 
                    "AI_Feedback": ai_feedback, 
                    "N": len(group_data),
                    "Mean": group_data.mean(),
                    "Std": group_data.std(),
                    "Shapiro-W": W, 
                    "Shapiro-p": p_shapiro,
                    "Normal": "Yes" if p_shapiro > 0.05 else "No"
                })
                groups_data.append(group_data)
            else:
                normality_results.append({
                    "Gender": gender, 
                    "AI_Feedback": ai_feedback, 
                    "N": len(group_data),
                    "Mean": np.nan,
                    "Std": np.nan,
                    "Shapiro-W": np.nan, 
                    "Shapiro-p": np.nan,
                    "Normal": "N/A"
                })
    
    norm_df = pd.DataFrame(normality_results)
    print(norm_df.to_string(index=False))
    
    # Overall assessment
    valid_tests = norm_df.dropna(subset=["Shapiro-p"])
    all_normal = (valid_tests["Shapiro-p"] > 0.05).all()
    print(f"\nAll groups pass normality test: {all_normal}")
    
    # Homogeneity of variance test (Levene's test)
    if len(groups_data) >= 2 and all(len(g) >= 2 for g in groups_data):
        levene_stat, levene_p = levene(*groups_data)
        print(f"Levene's test for equal variances: F={levene_stat:.4f}, p={levene_p:.4f}")
        equal_variances = levene_p > 0.05
        print(f"Equal variances assumption met: {equal_variances}")
        
        # Overall decision
        parametric_ok = all_normal and equal_variances
        print(f"\nParametric ANOVA appropriate: {parametric_ok}")
        return parametric_ok, norm_df
    else:
        print("Cannot perform Levene's test (insufficient data)")
        return False, norm_df

# Check normality for both phases
param_ok_train, norm_train = comprehensive_normality_check(df_train, "Total_Served", "Training")
param_ok_test, norm_test = comprehensive_normality_check(df_test, "Total_Served", "Testing")

# ================================================================
# 2) ALIGNED RANK TRANSFORM (ART) TEST
# ================================================================

def aligned_rank_transform_anova(data, dv, factor1, factor2):
    """
    Perform Aligned Rank Transform ANOVA for two-way factorial design.
    This is the proper non-parametric equivalent of two-way ANOVA.
    """
    print(f"\n=== ALIGNED RANK TRANSFORM ANOVA ===")
    print("This test allows for proper analysis of main effects and interactions")
    print("in a non-parametric two-way factorial design.\n")
    
    # Create a working copy and ensure DV is numeric
    df_work = data.copy()
    df_work[dv] = pd.to_numeric(df_work[dv], errors='coerce')
    
    # Step 1: Calculate marginal means for alignment
    # Overall mean
    grand_mean = df_work[dv].mean()
    
    # Main effect means
    factor1_means = df_work.groupby(factor1)[dv].mean()
    factor2_means = df_work.groupby(factor2)[dv].mean()
    
    # Cell means for interaction
    cell_means = df_work.groupby([factor1, factor2])[dv].mean()
    
    # Step 2: Create aligned scores for each effect
    # Convert categorical columns to string for mapping
    df_work[f'{factor1}_str'] = df_work[factor1].astype(str)
    df_work[f'{factor2}_str'] = df_work[factor2].astype(str)
    
    # Create mapping dictionaries with string keys
    factor1_means_dict = {str(k): v for k, v in factor1_means.items()}
    factor2_means_dict = {str(k): v for k, v in factor2_means.items()}
    cell_means_dict = {(str(k[0]), str(k[1])): v for k, v in cell_means.items()}
    
    # For main effect of factor1 (remove factor2 effect)
    df_work['factor2_mean'] = df_work[f'{factor2}_str'].map(factor2_means_dict)
    df_work['aligned_factor1'] = df_work[dv] - df_work['factor2_mean'] - grand_mean
    df_work['rank_factor1'] = df_work['aligned_factor1'].rank()
    
    # For main effect of factor2 (remove factor1 effect)
    df_work['factor1_mean'] = df_work[f'{factor1}_str'].map(factor1_means_dict)
    df_work['aligned_factor2'] = df_work[dv] - df_work['factor1_mean'] - grand_mean
    df_work['rank_factor2'] = df_work['aligned_factor2'].rank()
    
    # For interaction effect (remove both main effects)
    df_work['aligned_interaction'] = (df_work[dv] - 
                                    df_work['factor1_mean'] - 
                                    df_work['factor2_mean'] + 
                                    grand_mean)
    df_work['rank_interaction'] = df_work['aligned_interaction'].rank()
    
    # Step 3: Run ANOVA on ranked aligned scores
    print("Testing Main Effect of", factor1)
    anova_factor1 = pg.anova(dv='rank_factor1', between=[factor1], data=df_work)
    print(anova_factor1.to_string(index=False))
    
    print(f"\nTesting Main Effect of {factor2}")
    anova_factor2 = pg.anova(dv='rank_factor2', between=[factor2], data=df_work)
    print(anova_factor2.to_string(index=False))
    
    print(f"\nTesting Interaction Effect ({factor1} × {factor2})")
    anova_interaction = pg.anova(dv='rank_interaction', between=[factor1, factor2], data=df_work)
    # Get only the interaction row
    interaction_row = anova_interaction[anova_interaction['Source'] == f'{factor1} * {factor2}']
    print(interaction_row.to_string(index=False))
    
    return {
        'main_effect_1': anova_factor1,
        'main_effect_2': anova_factor2, 
        'interaction': interaction_row,
        'data_with_ranks': df_work
    }

# ================================================================
# 3) RUN ART ANALYSIS
# ================================================================

print("\n" + "="*60)
print("RUNNING ART ANALYSIS FOR TRAINING PHASE")
print("="*60)
art_train = aligned_rank_transform_anova(df_train, 'Total_Served', 'Gender', 'AI_Feedback')

print("\n" + "="*60)
print("RUNNING ART ANALYSIS FOR TESTING PHASE") 
print("="*60)
art_test = aligned_rank_transform_anova(df_test, 'Total_Served', 'Gender', 'AI_Feedback')

# ================================================================
# 4) POST-HOC ANALYSIS (if significant effects found)
# ================================================================

def art_posthoc_analysis(data, dv, factor1, factor2, art_results):
    """
    Perform post-hoc analysis for significant ART results
    """
    print(f"\n=== POST-HOC ANALYSIS ===")
    
    # Check if main effects or interaction are significant
    main1_sig = art_results['main_effect_1']['p-unc'].iloc[0] < 0.05
    main2_sig = art_results['main_effect_2']['p-unc'].iloc[0] < 0.05
    int_sig = art_results['interaction']['p-unc'].iloc[0] < 0.05 if len(art_results['interaction']) > 0 else False
    
    print(f"Significant main effect of {factor1}: {main1_sig}")
    print(f"Significant main effect of {factor2}: {main2_sig}")  
    print(f"Significant interaction: {int_sig}")
    
    if int_sig:
        print(f"\n--- Simple Effects Analysis (due to significant interaction) ---")
        # Test effect of factor2 within each level of factor1
        for level in data[factor1].cat.categories:
            subset = data[data[factor1] == level]
            if len(subset) > 0 and subset[factor2].nunique() > 1:
                group1 = subset[subset[factor2] == subset[factor2].cat.categories[0]][dv]
                group2 = subset[subset[factor2] == subset[factor2].cat.categories[1]][dv] 
                if len(group1) > 0 and len(group2) > 0:
                    u_stat, p_val = stats.mannwhitneyu(group1, group2, alternative='two-sided')
                    print(f"{factor2} effect within {factor1}={level}: U={u_stat:.2f}, p={p_val:.4f}")
    
    elif main2_sig and not int_sig:
        print(f"\n--- Main Effect Analysis for {factor2} ---")
        # Simple Mann-Whitney for the main effect
        levels = data[factor2].cat.categories
        if len(levels) == 2:
            group1 = data[data[factor2] == levels[0]][dv]
            group2 = data[data[factor2] == levels[1]][dv]
            u_stat, p_val = stats.mannwhitneyu(group1, group2, alternative='two-sided')
            print(f"{levels[0]} vs {levels[1]}: U={u_stat:.2f}, p={p_val:.4f}")
            
            # Effect size (r = Z / sqrt(N))
            z_score = stats.norm.ppf(p_val/2) * -1 if p_val < 1 else 0
            r = abs(z_score) / np.sqrt(len(data))
            print(f"Effect size (r): {r:.3f}")

# Run post-hoc for both phases
print("\n" + "="*50)
print("POST-HOC ANALYSIS - TRAINING PHASE")
print("="*50)
art_posthoc_analysis(df_train, 'Total_Failed', 'Gender', 'AI_Feedback', art_train)

print("\n" + "="*50)
print("POST-HOC ANALYSIS - TESTING PHASE")
print("="*50)
art_posthoc_analysis(df_test, 'Total_Failed', 'Gender', 'AI_Feedback', art_test)

# ================================================================
# 5) SUMMARY AND INTERPRETATION
# ================================================================

print("\n" + "="*60)
print("SUMMARY AND INTERPRETATION")
print("="*60)

def interpret_results(phase_name, art_results):
    print(f"\n--- {phase_name} Phase Results ---")
    
    # Extract p-values
    gender_p = art_results['main_effect_1']['p-unc'].iloc[0]
    ai_p = art_results['main_effect_2']['p-unc'].iloc[0]  
    int_p = art_results['interaction']['p-unc'].iloc[0] if len(art_results['interaction']) > 0 else 1.0
    
    print(f"Gender main effect: p = {gender_p:.4f} {'*' if gender_p < 0.05 else 'ns'}")
    print(f"AI Feedback main effect: p = {ai_p:.4f} {'*' if ai_p < 0.05 else 'ns'}")
    print(f"Gender × AI Feedback interaction: p = {int_p:.4f} {'*' if int_p < 0.05 else 'ns'}")
    
    # Interpretation
    if int_p < 0.05:
        print("→ The effect of AI feedback depends on gender (or vice versa)")
    elif ai_p < 0.05:
        print("→ AI feedback has a significant main effect on Total Served")
    elif gender_p < 0.05:
        print("→ Gender has a significant main effect on Total Served")
    else:
        print("→ No significant effects detected")

interpret_results("Training", art_train)
interpret_results("Testing", art_test)

print(f"\n{'*'*60}")
print("NOTE: ART (Aligned Rank Transform) is the appropriate test for your design")
print("because it properly handles non-parametric two-way factorial analysis")
print("while preserving the ability to test interactions.")
print(f"{'*'*60}")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
from scipy.stats import fligner, levene, bartlett
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your data is already loaded as df_train and df_test
# This code tests if variance in Total_Served differs by Gender and AI_Feedback

def comprehensive_variance_analysis(data, dv="Total_Served", factor1="Gender", factor2="AI_Feedback", phase_name=""):
    """
    Comprehensive analysis of variance differences across factorial groups
    """
    print(f"\n{'='*60}")
    print(f"VARIANCE ANALYSIS: {phase_name}")
    print(f"Testing if variability in {dv} differs by {factor1} and {factor2}")
    print(f"{'='*60}")
    
    # 1. Descriptive statistics by group
    print(f"\n--- DESCRIPTIVE STATISTICS ---")
    variance_desc = data.groupby([factor1, factor2])[dv].agg([
        'count', 'mean', 'std', 'var', 'min', 'max'
    ]).round(3)
    print(variance_desc)
    
    # Calculate coefficient of variation (CV) for each group
    cv_data = data.groupby([factor1, factor2])[dv].apply(
        lambda x: (x.std() / x.mean()) * 100 if x.mean() != 0 else np.nan
    ).round(2)
    print(f"\nCoefficient of Variation (CV %) by group:")
    print(cv_data)
    
    # 2. Extract groups for testing
    groups = []
    group_labels = []
    
    for g1 in data[factor1].unique():
        for g2 in data[factor2].unique():
            group_data = data[(data[factor1] == g1) & (data[factor2] == g2)][dv].dropna()
            if len(group_data) >= 2:  # Need at least 2 observations for variance
                groups.append(group_data)
                group_labels.append(f"{g1}_{g2}")
    
    print(f"\nNumber of groups with sufficient data: {len(groups)}")
    
    if len(groups) < 2:
        print("Insufficient groups for variance testing")
        return None
    
    # 3. VARIANCE HOMOGENEITY TESTS
    print(f"\n--- VARIANCE HOMOGENEITY TESTS ---")
    
    # Fligner-Killeen Test (most robust for non-normal data)
    try:
        fligner_stat, fligner_p = fligner(*groups)
        print(f"Fligner-Killeen Test:")
        print(f"  Statistic: {fligner_stat:.4f}")
        print(f"  p-value: {fligner_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if fligner_p < 0.05 else 'No significant variance differences'}")
        print(f"  Recommendation: Best for non-normal data ✓")
    except Exception as e:
        print(f"Fligner-Killeen Test failed: {e}")
        fligner_p = np.nan
    
    # Brown-Forsythe Test (robust version of Levene's)
    try:
        bf_stat, bf_p = levene(*groups, center='median')
        print(f"\nBrown-Forsythe Test (Levene's with median):")
        print(f"  Statistic: {bf_stat:.4f}")
        print(f"  p-value: {bf_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if bf_p < 0.05 else 'No significant variance differences'}")
        print(f"  Recommendation: Robust alternative ✓")
    except Exception as e:
        print(f"Brown-Forsythe Test failed: {e}")
        bf_p = np.nan
    
    # Traditional Levene's Test (with mean)
    try:
        levene_stat, levene_p = levene(*groups, center='mean')
        print(f"\nLevene's Test (traditional):")
        print(f"  Statistic: {levene_stat:.4f}")
        print(f"  p-value: {levene_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if levene_p < 0.05 else 'No significant variance differences'}")
        print(f"  Recommendation: Sensitive to non-normality ⚠️")
    except Exception as e:
        print(f"Levene's Test failed: {e}")
        levene_p = np.nan
    
    # Bartlett's Test (requires normality)
    try:
        bartlett_stat, bartlett_p = bartlett(*groups)
        print(f"\nBartlett's Test:")
        print(f"  Statistic: {bartlett_stat:.4f}")
        print(f"  p-value: {bartlett_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if bartlett_p < 0.05 else 'No significant variance differences'}")
        print(f"  Recommendation: Requires normal data ❌ (not recommended for your data)")
    except Exception as e:
        print(f"Bartlett's Test failed: {e}")
        bartlett_p = np.nan
    
    # 4. EFFECT SIZE AND PRACTICAL SIGNIFICANCE
    print(f"\n--- EFFECT SIZE ANALYSIS ---")
    
    # Variance ratio (largest/smallest variance)
    variances = [group.var() for group in groups]
    max_var = max(variances)
    min_var = min(variances)
    variance_ratio = max_var / min_var if min_var > 0 else np.inf
    
    print(f"Variance Ratio (max/min): {variance_ratio:.2f}")
    print(f"Interpretation:")
    if variance_ratio <= 2:
        print("  - Small difference in variability")
    elif variance_ratio <= 4:
        print("  - Moderate difference in variability")
    else:
        print("  - Large difference in variability")
    
    # Standard deviation ratio
    std_devs = [group.std() for group in groups]
    max_std = max(std_devs)
    min_std = min(std_devs)
    std_ratio = max_std / min_std if min_std > 0 else np.inf
    print(f"Standard Deviation Ratio (max/min): {std_ratio:.2f}")
    
    # 5. PAIRWISE COMPARISONS (if overall test is significant)
    if fligner_p < 0.05 or bf_p < 0.05:
        print(f"\n--- PAIRWISE VARIANCE COMPARISONS ---")
        print("Since overall test is significant, examining pairwise differences:")
        
        from itertools import combinations
        for i, (group1, label1) in enumerate(zip(groups, group_labels)):
            for j, (group2, label2) in enumerate(zip(groups, group_labels)):
                if i < j:  # Avoid duplicate comparisons
                    try:
                        f_stat, f_p = stats.levene(group1, group2, center='median')
                        print(f"  {label1} vs {label2}: F={f_stat:.3f}, p={f_p:.4f}")
                    except:
                        print(f"  {label1} vs {label2}: Could not compute")
    
    # 6. SUMMARY AND RECOMMENDATIONS
    print(f"\n--- SUMMARY ---")
    primary_test = "Fligner-Killeen" if not np.isnan(fligner_p) else "Brown-Forsythe"
    primary_p = fligner_p if not np.isnan(fligner_p) else bf_p
    
    print(f"Primary test result ({primary_test}): p = {primary_p:.4f}")
    
    if primary_p < 0.001:
        conclusion = "Very strong evidence of variance differences"
    elif primary_p < 0.01:
        conclusion = "Strong evidence of variance differences"
    elif primary_p < 0.05:
        conclusion = "Significant variance differences detected"
    else:
        conclusion = "No significant variance differences"
    
    print(f"Conclusion: {conclusion}")
    
    if primary_p < 0.05:
        print(f"\nPractical implications:")
        print(f"- Some groups show more/less consistency in {dv}")
        print(f"- Consider which factors contribute to increased variability")
        print(f"- May need different analysis strategies for high vs low variance groups")
    
    return {
        'fligner_p': fligner_p,
        'brown_forsythe_p': bf_p,
        'levene_p': levene_p,
        'bartlett_p': bartlett_p,
        'variance_ratio': variance_ratio,
        'std_ratio': std_ratio,
        'group_stats': variance_desc,
        'coefficient_variation': cv_data
    }

def plot_variance_analysis(data, dv="Total_Served", factor1="Gender", factor2="AI_Feedback", phase_name=""):
    """
    Create visualizations for variance analysis
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Variance Analysis: {phase_name}', fontsize=16)
    
    # 1. Box plot showing spread
    sns.boxplot(data=data, x=factor1, y=dv, hue=factor2, ax=axes[0,0])
    axes[0,0].set_title('Distribution by Groups (Box Plot)')
    axes[0,0].set_ylabel(dv)
    
    # 2. Violin plot showing distribution shape
    sns.violinplot(data=data, x=factor1, y=dv, hue=factor2, ax=axes[0,1])
    axes[0,1].set_title('Distribution Shape (Violin Plot)')
    axes[0,1].set_ylabel(dv)
    
    # 3. Standard deviation by group
    std_data = data.groupby([factor1, factor2])[dv].std().reset_index()
    std_pivot = std_data.pivot(index=factor1, columns=factor2, values=dv)
    sns.heatmap(std_pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1,0])
    axes[1,0].set_title('Standard Deviation by Group')
    
    # 4. Coefficient of Variation by group
    cv_data = data.groupby([factor1, factor2])[dv].apply(
        lambda x: (x.std() / x.mean()) * 100 if x.mean() != 0 else np.nan
    ).reset_index()
    cv_data.columns = [factor1, factor2, 'CV']
    cv_pivot = cv_data.pivot(index=factor1, columns=factor2, values='CV')
    sns.heatmap(cv_pivot, annot=True, fmt='.1f', cmap='Blues', ax=axes[1,1])
    axes[1,1].set_title('Coefficient of Variation (%) by Group')
    
    plt.tight_layout()
    plt.show()

# Example usage with your data:
print("TESTING VARIANCE DIFFERENCES IN YOUR FACTORIAL DESIGN")
print("=" * 70)

# Run analysis for both phases
if 'df_train' in globals():
    results_train = comprehensive_variance_analysis(df_train, "Total_Served", "Gender", "AI_Feedback", "Training Phase")
    plot_variance_analysis(df_train, "Total_Served", "Gender", "AI_Feedback", "Training Phase")

if 'df_test' in globals():
    results_test = comprehensive_variance_analysis(df_test, "Total_Served", "Gender", "AI_Feedback", "Testing Phase")
    plot_variance_analysis(df_test, "Total_Served", "Gender", "AI_Feedback", "Testing Phase")

print(f"\n{'*'*70}")
print("INTERPRETATION GUIDE:")
print("- Fligner-Killeen: Best test for your non-normal data")
print("- Brown-Forsythe: Good robust alternative")
print("- Variance Ratio >4: Large practical difference in variability")
print("- CV%: Coefficient of variation shows relative variability")
print(f"{'*'*70}")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import fligner, levene, bartlett
import matplotlib.pyplot as plt
import seaborn as sns

def variance_analysis_ai_only(data, dv="Total_Served", factor="AI_Feedback", phase_name=""):
    """
    Test if variance in the dependent variable differs between AI_Feedback conditions
    This is essentially a two-sample variance test
    """
    print(f"\n{'='*60}")
    print(f"VARIANCE ANALYSIS: {phase_name}")
    print(f"Testing if variability in {dv} differs by {factor}")
    print(f"{'='*60}")
    
    # 1. Descriptive statistics by AI_Feedback group
    print(f"\n--- DESCRIPTIVE STATISTICS ---")
    variance_desc = data.groupby(factor)[dv].agg([
        'count', 'mean', 'std', 'var', 'min', 'max'
    ]).round(3)
    print(variance_desc)
    
    # Calculate coefficient of variation (CV) for each group
    cv_data = data.groupby(factor)[dv].apply(
        lambda x: (x.std() / x.mean()) * 100 if x.mean() != 0 else np.nan
    ).round(2)
    print(f"\nCoefficient of Variation (CV %) by {factor}:")
    for condition, cv in cv_data.items():
        print(f"  {condition}: {cv}%")
    
    # 2. Extract the two groups
    ai_conditions = data[factor].unique()
    
    if len(ai_conditions) != 2:
        print(f"Error: Expected 2 AI conditions, found {len(ai_conditions)}")
        return None
    
    group1_data = data[data[factor] == ai_conditions[0]][dv].dropna()
    group2_data = data[data[factor] == ai_conditions[1]][dv].dropna()
    
    group1_name = ai_conditions[0]
    group2_name = ai_conditions[1]
    
    print(f"\nGroup sizes:")
    print(f"  {group1_name}: {len(group1_data)} participants")
    print(f"  {group2_name}: {len(group2_data)} participants")
    
    if len(group1_data) < 2 or len(group2_data) < 2:
        print("Insufficient data for variance testing (need at least 2 per group)")
        return None
    
    # 3. VARIANCE HOMOGENEITY TESTS
    print(f"\n--- VARIANCE EQUALITY TESTS ---")
    
    # F-test for equal variances (classical approach)
    var1 = group1_data.var()
    var2 = group2_data.var()
    
    # F-statistic (larger variance / smaller variance)
    f_stat = max(var1, var2) / min(var1, var2)
    
    # Degrees of freedom
    df1 = len(group1_data) - 1 if var1 >= var2 else len(group2_data) - 1
    df2 = len(group2_data) - 1 if var1 >= var2 else len(group1_data) - 1
    
    # Two-tailed p-value
    f_p_value = 2 * (1 - stats.f.cdf(f_stat, df1, df2))
    
    print(f"F-test for Equal Variances:")
    print(f"  F-statistic: {f_stat:.4f}")
    print(f"  Degrees of freedom: ({df1}, {df2})")
    print(f"  p-value: {f_p_value:.4f}")
    print(f"  Interpretation: {'Variances differ significantly' if f_p_value < 0.05 else 'No significant variance difference'}")
    print(f"  Assumption: Requires normal data ⚠️")
    
    # Levene's Test (more robust)
    try:
        levene_stat, levene_p = levene(group1_data, group2_data, center='mean')
        print(f"\nLevene's Test:")
        print(f"  Statistic: {levene_stat:.4f}")
        print(f"  p-value: {levene_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if levene_p < 0.05 else 'No significant variance difference'}")
        print(f"  Recommendation: More robust than F-test ✓")
    except Exception as e:
        print(f"Levene's Test failed: {e}")
        levene_p = np.nan
    
    # Brown-Forsythe Test (robust version using medians)
    try:
        bf_stat, bf_p = levene(group1_data, group2_data, center='median')
        print(f"\nBrown-Forsythe Test (Levene's with median):")
        print(f"  Statistic: {bf_stat:.4f}")
        print(f"  p-value: {bf_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if bf_p < 0.05 else 'No significant variance difference'}")
        print(f"  Recommendation: Most robust for non-normal data ✓✓")
    except Exception as e:
        print(f"Brown-Forsythe Test failed: {e}")
        bf_p = np.nan
    
    # Fligner-Killeen Test (non-parametric)
    try:
        fligner_stat, fligner_p = fligner(group1_data, group2_data)
        print(f"\nFligner-Killeen Test:")
        print(f"  Statistic: {fligner_stat:.4f}")
        print(f"  p-value: {fligner_p:.4f}")
        print(f"  Interpretation: {'Variances differ significantly' if fligner_p < 0.05 else 'No significant variance difference'}")
        print(f"  Recommendation: Non-parametric, best for your data ✓✓✓")
    except Exception as e:
        print(f"Fligner-Killeen Test failed: {e}")
        fligner_p = np.nan
    
    # 4. EFFECT SIZE AND PRACTICAL SIGNIFICANCE
    print(f"\n--- EFFECT SIZE ANALYSIS ---")
    
    std1 = group1_data.std()
    std2 = group2_data.std()
    
    # Variance ratio
    variance_ratio = var1 / var2 if var2 > 0 else np.inf
    print(f"Variance Ratio ({group1_name}/{group2_name}): {variance_ratio:.3f}")
    
    # Standard deviation ratio
    std_ratio = std1 / std2 if std2 > 0 else np.inf
    print(f"Standard Deviation Ratio ({group1_name}/{group2_name}): {std_ratio:.3f}")
    
    # Interpret effect size
    print(f"\nEffect Size Interpretation:")
    abs_var_ratio = max(variance_ratio, 1/variance_ratio) if variance_ratio > 0 else np.inf
    if abs_var_ratio <= 1.5:
        effect_size = "Small"
    elif abs_var_ratio <= 3:
        effect_size = "Medium"
    else:
        effect_size = "Large"
    
    print(f"  Variance difference magnitude: {effect_size}")
    
    if variance_ratio > 1:
        more_variable = group1_name
        less_variable = group2_name
    else:
        more_variable = group2_name
        less_variable = group1_name
    
    print(f"  More variable group: {more_variable}")
    print(f"  Less variable group: {less_variable}")
    
    # 5. PRACTICAL INTERPRETATION
    print(f"\n--- PRACTICAL IMPLICATIONS ---")
    
    # Determine which test result to use (prefer non-parametric)
    if not np.isnan(fligner_p):
        primary_test = "Fligner-Killeen"
        primary_p = fligner_p
    elif not np.isnan(bf_p):
        primary_test = "Brown-Forsythe"
        primary_p = bf_p
    else:
        primary_test = "Levene's"
        primary_p = levene_p
    
    print(f"Primary test result ({primary_test}): p = {primary_p:.4f}")
    
    if primary_p < 0.05:
        print(f"\n✓ SIGNIFICANT VARIANCE DIFFERENCE DETECTED")
        print(f"  - {more_variable} condition shows {'higher' if variance_ratio > 1 else 'lower'} variability")
        print(f"  - This suggests AI feedback affects consistency of performance")
        print(f"  - Consider why one condition leads to more variable outcomes")
        
        if 'AI_On' in [group1_name, group2_name] and 'AI_Off' in [group1_name, group2_name]:
            if (group1_name == 'AI_On' and variance_ratio > 1) or (group2_name == 'AI_On' and variance_ratio < 1):
                print(f"  - AI feedback INCREASES variability (less consistent performance)")
            else:
                print(f"  - AI feedback DECREASES variability (more consistent performance)")
    else:
        print(f"\n○ NO SIGNIFICANT VARIANCE DIFFERENCE")
        print(f"  - AI feedback conditions show similar variability")
        print(f"  - Treatment affects means but not consistency")
    
    # 6. SUMMARY STATISTICS TABLE
    print(f"\n--- SUMMARY TABLE ---")
    summary_stats = pd.DataFrame({
        'Condition': [group1_name, group2_name],
        'N': [len(group1_data), len(group2_data)],
        'Mean': [group1_data.mean(), group2_data.mean()],
        'Std_Dev': [std1, std2],
        'Variance': [var1, var2],
        'CV_%': [cv_data[group1_name], cv_data[group2_name]]
    }).round(3)
    
    print(summary_stats.to_string(index=False))
    
    return {
        'f_test_p': f_p_value,
        'levene_p': levene_p,
        'brown_forsythe_p': bf_p,
        'fligner_p': fligner_p,
        'variance_ratio': variance_ratio,
        'std_ratio': std_ratio,
        'effect_size': effect_size,
        'more_variable_group': more_variable,
        'summary_stats': summary_stats
    }

def plot_variance_comparison(data, dv="Total_Served", factor="AI_Feedback", phase_name=""):
    """
    Create visualizations comparing variance between AI conditions
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'Variance Comparison: {phase_name} - {factor} Effect', fontsize=16)
    
    # 1. Box plot
    sns.boxplot(data=data, x=factor, y=dv, ax=axes[0,0])
    axes[0,0].set_title('Distribution Comparison (Box Plot)')
    axes[0,0].set_ylabel(dv)
    
    # 2. Violin plot
    sns.violinplot(data=data, x=factor, y=dv, ax=axes[0,1])
    axes[0,1].set_title('Distribution Shape (Violin Plot)')
    axes[0,1].set_ylabel(dv)
    
    # 3. Standard deviation comparison
    std_data = data.groupby(factor)[dv].std()
    bars = axes[1,0].bar(std_data.index, std_data.values)
    axes[1,0].set_title('Standard Deviation by Condition')
    axes[1,0].set_ylabel('Standard Deviation')
    axes[1,0].set_xlabel(factor)
    
    # Add value labels on bars
    for bar, val in zip(bars, std_data.values):
        axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                      f'{val:.2f}', ha='center', va='bottom')
    
    # 4. Coefficient of Variation comparison
    cv_data = data.groupby(factor)[dv].apply(
        lambda x: (x.std() / x.mean()) * 100 if x.mean() != 0 else np.nan
    )
    bars = axes[1,1].bar(cv_data.index, cv_data.values, color='orange')
    axes[1,1].set_title('Coefficient of Variation (%)')
    axes[1,1].set_ylabel('CV (%)')
    axes[1,1].set_xlabel(factor)
    
    # Add value labels on bars
    for bar, val in zip(bars, cv_data.values):
        axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                      f'{val:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# Example usage with your data:
print("TESTING VARIANCE DIFFERENCES: AI FEEDBACK EFFECT ONLY")
print("=" * 65)

# Run analysis for both phases (assuming df_train and df_test exist)
if 'df_train' in globals():
    print("TRAINING PHASE ANALYSIS:")
    results_train = variance_analysis_ai_only(df_train, "Total_Served", "AI_Feedback", "Training Phase")
    plot_variance_comparison(df_train, "Total_Served", "AI_Feedback", "Training Phase")

if 'df_test' in globals():
    print("\nTESTING PHASE ANALYSIS:")
    results_test = variance_analysis_ai_only(df_test, "Total_Served", "AI_Feedback", "Testing Phase")
    plot_variance_comparison(df_test, "Total_Served", "AI_Feedback", "Testing Phase")

print(f"\n{'*'*65}")
print("INTERPRETATION GUIDE:")
print("• Fligner-Killeen Test: Best for non-normal data (RECOMMENDED)")
print("• Brown-Forsythe Test: Robust alternative using medians") 
print("• Variance Ratio > 1.5: Meaningful practical difference")
print("• CV%: Relative variability (accounts for different means)")
print("• Significant result = AI affects consistency, not just average")
print(f"{'*'*65}")